# EMBER end to end on Llama-3.1-8B-Instruct: §3.1 Sparse MF → §3.2 concept features → §3.3 embedding erasure

Paper: *Don't Forget Your Embeddings: Robust Knowledge Erasure via Precise Editing of Embeddings*
(Suslik, Shafran, Geva, EMNLP Findings 2026). This notebook runs the three stages of §3 for one concept,
using the paper's default hyperparameters. Each stage is split into small blocks, and each block starts
with a note on what it does.

| Step | Paper section | What happens | Repo code reused |
|---|---|---|---|
| **3.1** | Finding Embedding Features with Sparse MF | factorize $E_{V^*}^\top \approx ZY$ with WTA-sparse $Y$ | `SparseMatrixFactorization`, `fit_with_ridge` |
| **3.2** | Identifying Concept-Related Features | mass ratio $\rho_i \ge \tau$, then a two-stage Gemini judge → $F_C$ | `compute_embedding_stats`, prompts from `ember/interpret_features.py` |
| **3.3** | Erasing Concept-Related Features | $\mathbf e_t \leftarrow \mathbf e_t - \delta \sum_{i\in F_C} Y_{i,t}\,\mathbf z_i$ for concept tokens | same math as `ember/erasure/embed_edit.py` |

**Notation.** Paper $Z \in \mathbb R^{d\times k}$ = code `F_`; paper $Y \in \mathbb R^{k\times|V^*|}$ = code `G_.T`.
In this notebook, `Z` and `Y` use the paper's shapes.

**Requirements:** a GPU, `HF_TOKEN` with access to the gated `meta-llama/Llama-3.1-8B-Instruct`, and a Gemini API key for Step 3.2.

**What changes from the Gemma notebook** (`ember_three_steps_ancient_rome_fresh_sample.ipynb`), per the paper:

| | Gemma-2-2B-it | Llama-3.1-8B-Instruct | Source |
|---|---|---|---|
| $d$ | 2304 | 4096 (≈1.8× larger) | App. A.1 |
| rank $k$ | 100 | **200** (scaled with $d$; $s$, $\lambda$, $\|S\|$, $\tau$ unchanged) | App. A.1 |
| embedding scale | $\sqrt d$ applied in the forward pass (TL folds it into `W_E`) | **none**: TL `W_E` = HF `embed_tokens` | `embed_edit._embedding_scale` |
| $E$ / $U$ | tied, so `load_hf_model` unties `lm_head` | **untied** already; only $E$ is edited, as in the paper | §6 |
| $\delta$ behaviour | efficacy rises fast up to $\delta\approx 50$ | efficacy saturates at $\delta\approx 50$–$100$; specificity drops sharply from $\delta\approx 50$ | App. C.4, Fig. 11 |
| edited tokens | at most 31–86 (≤0.034% of $V$) | at most 22–174 (≤0.136% of $V$) | §5.2 |

Because the model is 4× larger, this notebook also loads the TransformerLens copy in bf16 (fp32 would need ~32 GB plus the HF copy TL loads first), and the `erased` / `tuned` models share every tensor except the input embedding matrix with `original_model` (≈1 GB each instead of ≈16 GB).

---
## 0. Setup

### 0.1 Hyperparameters (paper defaults)

| Symbol | Value | Source |
|---|---|---|
| $k$ (rank) | 100 for Gemma-2-2B-it, 200 for Llama-3.1-8B-Instruct | App. A.1 |
| $s$ (WTA sparsity on $Y$) | 0.01, so each factor keeps $\lceil 0.01\,|V^*| \rceil$ tokens | App. A.1 |
| $\lambda$ (ridge) | $10^{-4}$ | App. A.1 |
| $T$ / patience / tol | 20,000 iters / 500 iters / $10^{-4}$ | App. A.1 |
| $\|S_C\| = \|S_N\|$ | 300 sentences each (`data/*_sentences.json`) | §4.1 |
| $\tau$ (mass-ratio threshold) | 2.0 | App. A.3 |
| judge confidence | ≥ 0.85, with 20 tokens per feature | App. A.3 |
| $\delta$ (edit strength) | tuned per concept over {0.5, 1, 2, 5, 10, 50, 100, 200} | App. C.3 |

In [29]:
import gc
import json
import math
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()  # HF_TOKEN / GEMINI_API_KEY from .env, if present
pd.set_option("display.max_colwidth", 200)

# ---- what to erase ------------------------------------------------------
CONCEPT = "Ancient Rome"
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"     # gated: needs HF_TOKEN with access
SEED = 42

# ---- Step 3.1: Sparse MF (App. A.1) -------------------------------------
RANK = 100 # 200 if "llama" in MODEL_NAME.lower() else 100   # k
SPARSITY = 0.01          # s
RIDGE = 1e-4             # lambda
MAX_ITER = 20_000        # T
PATIENCE = 200

# ---- Step 3.2: feature selection (App. A.3) -----------------------------
TAU = 2.0
CONFIDENCE = 0.85
TOP_K_TOKENS = 20
GEMINI_MODEL = "models/gemini-3.5-flash-lite"   # repo default; the paper used Gemini-2.5-Flash-Lite

# ---- Step 3.3: erasure (App. C.3) ---------------------------------------
DELTA_GRID = [0.5, 1.0, 2.0, 5.0, 10.0, 50.0, 100.0, 200.0]
DELTA = 10.0             # not tuned for Llama / Ancient Rome; on Llama efficacy saturates around 50-100 (Fig. 11), see 3.4

concepts = [e["concept"] for e in json.loads(Path("data/concept_sentences.json").read_text())]
assert CONCEPT in concepts, f"{CONCEPT!r} is not one of: {concepts}"
assert os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN"), "Llama-3.1 is gated: set HF_TOKEN (e.g. in .env)"
print(f"concept={CONCEPT!r} | model={MODEL_NAME} | k={RANK}")

concept='Ancient Rome' | model=meta-llama/Llama-3.1-8B-Instruct | k=100


### 0.2 Gemini API key

Step 3.2 is the only step that calls Gemini. You can paste your key below or put `GEMINI_API_KEY=...`
in `.env`. Clear the key from this cell before you commit or share the notebook.

In [30]:
GEMINI_API_KEY = ""  # paste here, or leave empty to use .env / the environment

if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
assert any(os.getenv(k) for k in ("GOOGLE_API_KEY", "GEMINI_API_TOKEN", "GEMINI_API_KEY")), \
    "No Gemini key found: Step 3.2 needs one."
print("Gemini key found.")

Gemini key found.


### 0.3 Repo imports

These are the building blocks the paper's pipeline uses. Reusing them means the notebook gives the same
results as `ember.train_mf_features`, `ember.interpret_features`, and `ember.run_erasure`.

In [31]:
from llm_utils.activation_generator import ActivationGenerator, extract_token_ids_sample_ids_and_labels
from transformers import AutoTokenizer

from ember.erasure.model_loader import load_hf_model
from ember.evals.mc import evaluate_mc_generation, prepare_mc_items
from ember.evals.model_wrap import WrappedHFModel
from ember.interpret_features import (
    MEMBERSHIP_PROMPT,
    TOKENS_DESC_SYS,
    SimpleGeminiClient,
    classify_membership_with_desc,
    get_description,
)
from ember.local_datasets import ConceptDataset, load_mc_qa_items, load_open_qa_examples
from ember.timing import Timer
from ember.utils import (
    SparseMatrixFactorization,
    _safe_model_name,
    build_token_label_codes,
    collect_feature_rows_for_embeddings,
    compute_embedding_stats,
    fit_with_ridge,
    get_embedding_matrix,
    get_special_token_ids,
    resolve_device,
    set_seed,
)

set_seed(SEED)
# DEVICE = resolve_device("auto")
# This host's vGPU reports CUDA as available but prohibits compute, so force CPU.
DEVICE = "cpu"
os.environ["EMBER_DEVICE"] = "cpu"  # load_hf_model reads this
print(f"device: {DEVICE}")

device: cpu


---
# Step 3.1: Finding embedding features with Sparse Matrix Factorization

**Goal:** split each token embedding $\mathbf e_t$ into a sum of a few shared directions, so that
semantically related tokens (e.g. *Harry*, *Hogwarts*, *wand*) share a feature:

$$E_{V^*}^\top \approx Z\,Y,\qquad Z\in\mathbb R^{d\times k}\ \text{(dense features)},\quad Y\in\mathbb R^{k\times|V^*|}\ \text{(WTA-sparse coefficients)}$$

The hard winner-takes-all (WTA) constraint on the rows of $Y$ means each feature $\mathbf z_i$ is used by only
1% of the tokens in $V^*$. That is why a feature can be specific to one concept.

### 3.1.a Load the model (TransformerLens)

The pipeline reads $E$ through TransformerLens. Unlike Gemma, Llama has no $\sqrt{d}$ embedding multiplier,
and TL does not center `W_E` for RMSNorm models, so TL's `W_E` is exactly HF's `embed_tokens` and Step 3.3
writes back with scale 1.

**Memory.** `ActivationGenerator` loads TL in fp32 (~32 GB for 8B, after first loading the HF weights), which does
not fit next to the later HF copies. The subclass below loads it in bf16, the dtype the HF checkpoint is stored in,
so $E$ is the same values the HF model uses.

In [32]:
from transformer_lens import HookedTransformer


class BF16ActivationGenerator(ActivationGenerator):
    """ActivationGenerator whose TL model is loaded in bf16 instead of fp32."""

    def __init__(self, model_name, model_device="cpu", data_device="cpu", mode="mlp"):
        self.model = HookedTransformer.from_pretrained(model_name, device=model_device, dtype=torch.bfloat16)
        self.model_name = model_name
        self.data_device = data_device
        self._mode = mode


act_generator = BF16ActivationGenerator(MODEL_NAME, model_device=DEVICE, data_device="cpu", mode="mlp")
tl_model = act_generator.model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

E = get_embedding_matrix(tl_model)  # (|V|, d)
VOCAB_SIZE, D_MODEL = E.shape
print(f"E: {tuple(E.shape)}  dtype={E.dtype}")

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 73.48it/s]


Loaded pretrained model meta-llama/Llama-3.1-8B-Instruct into HookedTransformer
E: (128256, 4096)  dtype=torch.bfloat16


### 3.1.b Build $S_C \cup S_N$ and the token set $V^*$

- $S_C$: 300 Wikipedia sentences about the concept. $S_N$: 300 neutral Wikipedia sentences.
  **Both are resampled on every run.** $S_C$ is drawn from the live "Ancient Rome" article and $S_N$ takes one
  sentence from each of 300 random pages of the local wikimedia/wikipedia sample (`wiki_dump_pages.jsonl`).
  The run's seed is printed and the sample is saved under `experiments/resample_sentences/data/runs/seed_<seed>/`;
  set `SAMPLE_SEED` to that number to rerun on the same sentences.
- $V^*$: the unique non-special token ids in $S_C \cup S_N$. The model factorizes only these rows of $E$,
  which is cheaper and keeps the factorization focused on the concept. The neutral tokens let Step 3.2
  tell concept features apart from general ones.

In [33]:
# Fresh S_C / S_N on every run, built with the helpers of experiments/resample_sentences/fetch_en_sentences.py.
import random
import sys

sys.path.insert(0, "experiments/resample_sentences")
from fetch_en_sentences import neutral_sentences, page_paragraphs, split_sentences

SAMPLE_SEED = None   # None draws a new seed; set it to a printed seed to reuse that run's sentences
N_SAMPLE = 300
RESAMPLED = Path("experiments/resample_sentences/data")

if SAMPLE_SEED is None:
    SAMPLE_SEED = random.SystemRandom().randrange(2**31)
run_dir = RESAMPLED / "runs" / f"seed_{SAMPLE_SEED}"
concept_path, neutral_path = run_dir / "concept_sentences.json", run_dir / "neutral_sentences.json"

if concept_path.exists() and neutral_path.exists():
    print(f"[sample] reusing {run_dir}")
else:
    canon, revid, paras = page_paragraphs(CONCEPT)
    pool = split_sentences(paras)
    concept_sents = random.Random(SAMPLE_SEED).sample(pool, min(N_SAMPLE, len(pool)))
    neutral = neutral_sentences(RESAMPLED / "wiki_dump_pages.jsonl", N_SAMPLE, random.Random(SAMPLE_SEED), exclude=set())
    run_dir.mkdir(parents=True, exist_ok=True)
    concept_path.write_text(json.dumps([{
        "concept": CONCEPT, "source": "https://en.wikipedia.org/wiki/" + canon.replace(" ", "_"),
        "revid": revid, "seed": SAMPLE_SEED, "sentences": concept_sents,
    }], ensure_ascii=False, indent=1))
    neutral_path.write_text(json.dumps(neutral, ensure_ascii=False, indent=1))
    print(f"[sample] '{canon}' rev {revid}: {len(concept_sents)} of {len(pool)} concept sentences, "
          f"{len(neutral)} neutral -> {run_dir}")
print(f"SAMPLE_SEED = {SAMPLE_SEED}")

dataset = ConceptDataset(concept_name=CONCEPT, neutral_sample_seed=42,
                         concept_path=concept_path, neutral_path=neutral_path)
token_ids_all, sample_ids, labels_all = extract_token_ids_sample_ids_and_labels(dataset, act_generator)

special = get_special_token_ids(tl_model)
vstar_ids = sorted({int(t) for t in token_ids_all.tolist()} - special)

n_concept_sents = sum(lbl == CONCEPT for _, lbl in dataset)
print(f"|S_C| = {n_concept_sents}   |S_N| = {len(dataset) - n_concept_sents}")
print(f"token occurrences = {len(token_ids_all)}  ->  |V*| = {len(vstar_ids)} "
      f"({len(vstar_ids) / VOCAB_SIZE:.2%} of the vocabulary)")

[sample] 'Ancient Rome' rev 1379589331: 300 of 710 concept sentences, 300 neutral -> experiments/resample_sentences/data/runs/seed_15630710
SAMPLE_SEED = 15630710


Extracting token IDs: 100%|██████████| 120/120 [00:00<00:00, 859.20it/s]

|S_C| = 300   |S_N| = 300
token occurrences = 16696  ->  |V*| = 4973 (3.88% of the vocabulary)


### 3.1.c Build the data matrix $A = E_{V^*}^\top$

Each column of $A$ is one token's embedding.

In [34]:
E_vstar = E.index_select(0, torch.tensor(vstar_ids, device=E.device)).detach().float()  # (|V*|, d)
A = E_vstar.T.contiguous().to(DEVICE)                                                      # (d, |V*|)

print(f"A: {tuple(A.shape)}  (d x |V*|)")
print(f"WTA keeps ceil(s*|V*|) = {math.ceil(SPARSITY * len(vstar_ids))} tokens per feature")

A: (4096, 4973)  (d x |V*|)
WTA keeps ceil(s*|V*|) = 50 tokens per feature


### 3.1.d Fit Sparse MF (alternating least squares + WTA)

The fit alternates two closed-form ridge updates:

$$Z \leftarrow A Y^\top (Y Y^\top + \lambda I)^{-1},\qquad Y \leftarrow (Z^\top Z + \lambda I)^{-1} Z^\top A$$

After each $Y$ update it applies WTA, which keeps the top-$s$ fraction of $|Y_{i,:}|$ and zeroes the rest.
It also rescales between the two factors. Initialization is $\mathcal N(0,1)$, and the fit stops early after
500 iterations without an improvement of more than $10^{-4}$. If the matrix inverse is singular,
`fit_with_ridge` retries with $\lambda \times 10$.

This is the slowest cell in Step 3.1. It prints progress every 50 iterations.

In [35]:
set_seed(SEED)
smf = SparseMatrixFactorization(RANK, fitting_device=DEVICE, g_sparsity=SPARSITY)
with Timer() as t_fit:
    ridge_used = fit_with_ridge(smf, A, MAX_ITER, patience=PATIENCE, base_reg=RIDGE)

Z = smf.F_.detach().float().cpu()      # (d, k)      embedding features z_i (columns)
Y = smf.G_.detach().float().cpu().T    # (k, |V*|)   sparse coefficients
print(f"\nfitted in {t_fit['elapsed']:.1f}s (lambda={ridge_used:g})  Z: {tuple(Z.shape)}  Y: {tuple(Y.shape)}")

Iter    0: loss=2090.460205  (best=2090.460205, no_improve=0)
Iter   50: loss=1907.697144  (best=1907.697144, no_improve=0)
Iter   89: loss=1906.626953  (best=1906.622314, no_improve=1)
Iter  100: loss=1906.560669  (best=1906.560669, no_improve=0)
Iter  107: loss=1906.528320  (best=1906.525757, no_improve=1)
Iter  112: loss=1906.528687  (best=1906.515381, no_improve=1)
Iter  131: loss=1906.512695  (best=1906.511353, no_improve=1)
Iter  150: loss=1906.520752  (best=1906.511353, no_improve=20)
Iter  200: loss=1906.520386  (best=1906.511353, no_improve=70)
Iter  250: loss=1906.520386  (best=1906.511353, no_improve=120)
Iter  300: loss=1906.520386  (best=1906.511353, no_improve=170)
Stopping early at iter 330 (no improvement in 200 iters)

fitted in 20.0s (lambda=0.0001)  Z: (4096, 100)  Y: (100, 4973)


### 3.1.e Check the factorization

Reconstruction quality and sparsity. Each feature should use ≈ $\lceil s|V^*|\rceil$ tokens. Tokens that
no feature covers are left as pure reconstruction error $\boldsymbol\varepsilon_t$.

In [36]:
A_cpu = A.cpu()
rel_err = (torch.norm(A_cpu - Z @ Y) / torch.norm(A_cpu)).item()
nz = Y.abs() > 0
tokens_per_feature = nz.sum(1)
features_per_token = nz.sum(0)

print(f"relative reconstruction error ||A - ZY||_F / ||A||_F = {rel_err:.4f}")
print(f"tokens per feature : min={tokens_per_feature.min().item()}  max={tokens_per_feature.max().item()}")
print(f"features per token : mean={features_per_token.float().mean():.2f}  max={features_per_token.max().item()}  "
      f"uncovered tokens={int((features_per_token == 0).sum())}")

relative reconstruction error ||A - ZY||_F / ||A||_F = 0.9508
tokens per feature : min=50  max=50
features per token : mean=1.01  max=7  uncovered tokens=1569


In [37]:
print(Z.size())
print(Y.size())
print(RANK)

torch.Size([4096, 100])
torch.Size([100, 4973])
100


### 3.1.f Baseline: truncated SVD with the same $k$

The rank-$k$ truncated SVD $A_k = U_k \Sigma_k V_k^\top$ is the best possible rank-$k$ approximation
(Eckart–Young theorem), so no rank-$k$ factorization can have a lower error, including Sparse MF.
The gap between the two errors is what the WTA sparsity costs in reconstruction.

In [38]:
U, S, Vh = torch.linalg.svd(A_cpu, full_matrices=False)
A_svd = U[:, :RANK] @ torch.diag(S[:RANK]) @ Vh[:RANK]
svd_rel_err = (torch.norm(A_cpu - A_svd) / torch.norm(A_cpu)).item()

svd_nnz = U[:, :RANK].numel() + RANK + Vh[:RANK].numel()
smf_nnz = Z.numel() + int(nz.sum())
print(f"rank-{RANK} truncated SVD : relative error = {svd_rel_err:.4f}   (parameters: {svd_nnz:,})")
print(f"rank-{RANK} Sparse MF     : relative error = {rel_err:.4f}   (nonzero parameters: {smf_nnz:,})")

# Smallest SVD rank whose error is no worse than Sparse MF's.
tail = torch.sqrt(torch.flip(torch.cumsum(torch.flip(S ** 2, [0]), 0), [0]) / (S ** 2).sum())
matching_rank = int((tail <= rel_err).nonzero()[0]) if (tail <= rel_err).any() else len(S)
print(f"SVD reaches the Sparse MF error ({rel_err:.4f}) at rank {matching_rank}")

rank-100 truncated SVD : relative error = 0.9025   (parameters: 907,000)
rank-100 Sparse MF     : relative error = 0.9508   (nonzero parameters: 414,600)
SVD reaches the Sparse MF error (0.9508) at rank 22


---
# Step 3.2: Identifying concept-related features

Most of the $k$ features are general (punctuation, function words, and so on). The selection runs in two filters:

1. **Mass ratio** $\rho_i$: is feature $i$ used mostly by concept tokens rather than neutral tokens? Keep the feature if $\rho_i \ge \tau = 2.0$.
2. **LLM judge** (Gemini, two stages): describe the feature's tokens *without* seeing the concept name,
   then ask whether that description is the concept. Keep the feature if `is_member` is true and confidence ≥ 0.85.

The features that pass both filters form $F_C$.

### 3.2.a Label every token in $V^*$

$\ell_t \in$ {concept, neutral, both}, depending on whether $t$ appears in $S_C$ only, $S_N$ only, or both.
Codes: `1` = concept, `0` = neutral, `2` = both.

In [39]:
token_to_code = build_token_label_codes(CONCEPT, dataset, tokenizer)
codes = np.array([token_to_code.get(t, 0) for t in vstar_ids])

print(f"T_C (concept-only) = {(codes == 1).sum()}   T_N (neutral-only) = {(codes == 0).sum()}   "
      f"both = {(codes == 2).sum()}")

T_C (concept-only) = 1746   T_N (neutral-only) = 2479   both = 748


### 3.2.b Score features by mass ratio and keep $\rho_i \ge \tau$

As implemented in `compute_embedding_stats` (column `ratio_abs`):

$$\rho_i = \frac{\operatorname{mean}_{t\in T_C} |Y_{i,t}|}{\operatorname{mean}_{t\in T_N} |Y_{i,t}|}$$

Tokens labeled *both* are excluded from the ratio. The threshold sweep below is the per-concept view of
Figure 6 in the paper.

In [40]:
stats = compute_embedding_stats(Y.T, vstar_ids, token_to_code, CONCEPT).sort_values("ratio_abs", ascending=False)

for thr in [1.0, 1.25, 1.5, 1.75, 2.0, 2.25]:
    print(f"rho >= {thr:<4}: {(stats['ratio_abs'] >= thr).sum():3d} / {RANK} features")

candidates = stats[stats["ratio_abs"] >= TAU]
print(f"\n{len(candidates)} candidate features pass tau = {TAU}")
candidates[["feature", "ratio_abs", "mean_abs_concept", "mean_abs_neutral", "num_concept", "num_neutral", "num_both"]]

rho >= 1.0 :  64 / 100 features
rho >= 1.25:  52 / 100 features
rho >= 1.5 :  44 / 100 features
rho >= 1.75:  39 / 100 features
rho >= 2.0 :  35 / 100 features
rho >= 2.25:  32 / 100 features

35 candidate features pass tau = 2.0


,feature,ratio_abs,mean_abs_concept,mean_abs_neutral,num_concept,num_neutral,num_both
84,84,6.314508,0.002404,0.000381,35,13,2
21,21,6.099844,0.002702,0.000443,31,11,8
22,22,5.401013,0.001858,0.000344,30,7,13
43,43,5.263011,0.002294,0.000436,29,11,10
96,96,5.027533,0.001814,0.000361,23,7,20
40,40,4.966961,0.002594,0.000522,32,11,7
55,55,4.623705,0.001655,0.000358,21,8,21
19,19,4.482253,0.002291,0.000511,29,10,11
23,23,4.349009,0.002471,0.000568,32,13,5
29,29,4.149677,0.002354,0.000567,34,14,2


### 3.2.c Collect the tokens the judge will see

For each candidate, the code takes the tokens with nonzero $Y_{i,t}$, sorted by $|Y_{i,t}|$, and keeps the top 20.
The paper text says "concept-labeled tokens", but the released code sends the top activating tokens
whatever their label. This notebook follows the code. The `labels` column shows each token's label.

In [41]:
token_label_map = {tid: "both" if c == 2 else CONCEPT if c == 1 else "Neutral" for tid, c in token_to_code.items()}
feature_rows = collect_feature_rows_for_embeddings(
    Y.T, vstar_ids, token_label_map, tl_model, _safe_model_name(MODEL_NAME), CONCEPT, RANK,
)
feat_tokens = pd.DataFrame(feature_rows).set_index("feature")[["activating_tokens", "labels"]]

cand = candidates[["feature", "ratio_abs"]].join(feat_tokens, on="feature").reset_index(drop=True)
cand["judge_tokens"] = cand["activating_tokens"].apply(lambda toks: toks[:TOP_K_TOKENS])
cand[["feature", "ratio_abs", "judge_tokens"]]

,feature,ratio_abs,judge_tokens
0,84,6.314508,"[ invading, invaded, invasion, invaders, siege, conquered, asions, inv, occupying, inception, conquest, attacked, treaties, barric, attack, intervention, revolt, impoverished, int..."
1,21,6.099844,"[305, 306, 303, 312, 313, 270, 275, 330, 335, 408, 284, 375, 300, 286, 410, 264, 480, 338, 271, 324]"
2,22,5.401013,"[ commonly, rarely, often, frequently, usually, sometimes, common, widely, generally, never, traditionally, known, needed, known, popular, worried, forbidden, important, worn, ever]"
3,43,5.263011,"[ armies, army, navy, infantry, troops, Army, naval, military, artillery, soldiers, Military, forces, stationed, militar, Guards, warfare, fleets, generals, ships, Force]"
4,96,5.027533,"[Although, While, Despite, During, When, After, Because, With, However, But, Even, Before, Along, If, In, Throughout, There, According, These, Following]"
5,40,4.966961,"[ restored, abolished, reclaimed, rebuilt, preserved, retained, renewed, restore, replaced, demolished, removed, abandoned, suppressed, discontinued, restrained, reduced, retaining..."
6,55,4.623705,"[ dozens, hundreds, numerous, several, many, thousands, Several, various, some, Many, some, Some, few, dozen, fewer, any, others, plentiful, certain, another]"
7,19,4.482253,"[ morality, moral, sexuality, culture, social, political, traditions, sexual, literary, cultures, religious, Culture, religion, tradition, Social, politically, classical, ethics, ..."
8,23,4.349009,"[ renowned, esteemed, reputation, legendary, prized, celebrated, praised, prestige, hailed, prominent, known, wealthy, regarded, known, controversial, precious, -established, distin..."
9,29,4.149677,"[ dictatorship, dictator, totalitarian, authoritarian, monarchy, unconstitutional, patriarch, confrontation, persecution, archy, archs, rhetoric, populist, thriving, proletariat, porn..."


### 3.2.d-0 Rate-limited Gemini client

The free tier allows **15 requests per minute per model**. Stage 1 and Stage 2 make one request per
candidate each, so 2 × (number of candidates) requests in total. This wrapper does three things:

- spaces requests at least `60 / RPM` seconds apart (with a 10% margin);
- on a per-minute 429, waits the `retry in Ns` delay the server reports, then retries;
- fails immediately on a *daily* quota error, because waiting would not help.

Answers are cached in `descriptions` and `verdicts`. Re-running a stage only queries the features that
are still missing, so a crash never wastes quota you already spent.

In [42]:
import re
import time

GEMINI_RPM = 15


class ThrottledGemini:
    def __init__(self, inner: SimpleGeminiClient, rpm: int = GEMINI_RPM, max_attempts: int = 6):
        inner.max_retries = 1  # retries are handled here, with the server's retry delay
        self.inner = inner
        self.interval = 60.0 / rpm * 1.1
        self.max_attempts = max_attempts
        self._last = 0.0

    def generate(self, prompt: str) -> str:
        for attempt in range(self.max_attempts):
            time.sleep(max(0.0, self._last + self.interval - time.monotonic()))
            self._last = time.monotonic()
            try:
                return self.inner.generate(prompt)
            except RuntimeError as e:
                msg = str(e)
                if "429" not in msg or "PerDay" in msg or attempt == self.max_attempts - 1:
                    raise
                m = re.search(r"retry in ([\d.]+)s", msg)
                time.sleep(float(m.group(1)) + 1 if m else 60)


USE_THROTTLE = False  # billing enabled; set True to go back to free-tier pacing

if USE_THROTTLE:
    client = ThrottledGemini(SimpleGeminiClient(model_name=GEMINI_MODEL))
else:
    client = SimpleGeminiClient(model_name=GEMINI_MODEL)

# Kept across re-runs; call .clear() on both after re-fitting Step 3.1.
descriptions = globals().get("descriptions", {})
verdicts = globals().get("verdicts", {})
print(f"{2 * len(cand)} requests needed, {len(descriptions)} descriptions and {len(verdicts)} verdicts cached; "
      f"throttle={'on' if USE_THROTTLE else 'off'}")

70 requests needed, 64 descriptions and 64 verdicts cached; throttle=off


### 3.2.d LLM judge, stage 1: describe each feature *without* the concept name

Stage 1 never sees the concept name. This avoids confirmation bias, where tokens like *wizard* could
look like "Harry Potter" just because the judge was told the concept.

In [43]:
print("---- Stage 1 prompt ----")
print(TOKENS_DESC_SYS.strip(), "\n\nTokens: {tokens}\nExplanation of feature behavior:")

todo = cand[~cand["feature"].astype(int).isin(descriptions)]
for _, row in tqdm(todo.iterrows(), total=len(todo), desc="Stage 1: describe"):
    descriptions[int(row["feature"])] = get_description(client, row["judge_tokens"])
cand["description"] = cand["feature"].map(descriptions)
cand[["feature", "ratio_abs", "description"]]

---- Stage 1 prompt ----
You get tokens that represent a single feature vector (with some noise).
Infer the single most specific, cohesive concept shared by the relevant tokens.
Return only one concise sentence. No preface, no list, no caveats.

Example:
Tokens: ['▁tomorrow','▁tonight','▁yesterday','▁today','▁demain']
Explanation of feature behavior: this vector is related to specific dates and times (e.g., today/tomorrow/yesterday). 

Tokens: {tokens}
Explanation of feature behavior:


Stage 1: describe: 100%|██████████| 25/25 [00:33<00:00,  1.33s/it]


,feature,ratio_abs,description
0,84,6.314508,"This feature vector represents military conquest, invasion, and hostile territorial occupation."
1,21,6.099844,This vector represents numeric values typically associated with area codes or measurement quantities.
2,22,5.401013,"This vector represents historical conflict, governance, and societal upheaval."
3,43,5.263011,"This vector represents armed forces and military personnel, equipment, and operations."
4,96,5.027533,This feature vector activates on subordinating and coordinating conjunctions or transitional adverbs used to introduce clauses or establish logical relationships.
5,40,4.966961,"This feature represents action verbs associated with acquiring, maintaining, or carrying out processes and achievements."
6,55,4.623705,These tokens represent indefinite quantities or plural amounts.
7,19,4.482253,"This vector represents human societal institutions, belief systems, and cultural norms."
8,23,4.349009,"This vector represents the concept of fame, high status, and widespread public acclaim."
9,29,4.149677,"These tokens represent a feature vector associated with high social status, recognition, and distinction."


In [44]:
N_SHOW = 10
LABEL_TAG = {1: "C", 0: "N", 2: "B"}  # concept / neutral / both

for _, row in cand.sort_values("ratio_abs", ascending=False).iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini : {descriptions.get(feat, '<not described yet>')}")
    print("  tokens : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

=== feature 84  rho=6.31 ===
  Gemini : This feature vector represents military conquest, invasion, and hostile territorial occupation.
  tokens : 'Ġinvading'[C] 0.44, 'Ġinvaded'[C] 0.44, 'Ġinvasion'[C] 0.40, 'Ġinvaders'[C] 0.36, 'Ġsiege'[C] 0.16, 'Ġconquered'[C] 0.16, 'asions'[C] 0.15, 'Ġinv'[C] 0.13, 'Ġoccupying'[N] 0.12, 'Ġinception'[N] 0.12

=== feature 21  rho=6.10 ===
  Gemini : This vector represents numeric values typically associated with area codes or measurement quantities.
  tokens : '305'[N] 0.22, '306'[C] 0.22, '303'[C] 0.21, '312'[C] 0.20, '313'[C] 0.20, '270'[C] 0.19, '275'[C] 0.19, '330'[C] 0.19, '335'[N] 0.19, '408'[C] 0.18

=== feature 22  rho=5.40 ===
  Gemini : This vector represents historical conflict, governance, and societal upheaval.
  tokens : 'Ġcommonly'[C] 0.36, 'Ġrarely'[C] 0.32, 'Ġoften'[B] 0.32, 'Ġfrequently'[N] 0.31, 'Ġusually'[B] 0.25, 'Ġsometimes'[B] 0.25, 'Ġcommon'[B] 0.18, 'Ġwidely'[C] 0.17, 'Ġgenerally'[C] 0.16, 'Ġnever'[B] 0.14

=== feature 43  rh

### 3.2.e LLM judge, stage 2: is the description *the* concept?

Stage 2 sees only the concept name and the Stage 1 description. It returns `{"is_member", "confidence"}`.
A feature joins $F_C$ only if `is_member` is true **and** confidence ≥ 0.85.

In [45]:
print("---- Stage 2 prompt ----")
print(MEMBERSHIP_PROMPT.format(concept=CONCEPT, desc="{description}").strip())

todo = {f: d for f, d in descriptions.items() if f not in verdicts}
for feat, desc in tqdm(todo.items(), desc="Stage 2: classify"):
    verdicts[feat] = classify_membership_with_desc(client, CONCEPT, desc)

cand["is_member"] = cand["feature"].map(lambda f: verdicts[int(f)][0])
cand["confidence"] = pd.to_numeric(cand["feature"].map(lambda f: verdicts[int(f)][1]), errors="coerce").fillna(0.0)
cand["selected"] = (cand["is_member"] == True) & (cand["confidence"] >= CONFIDENCE)

F_C = sorted(cand.loc[cand["selected"], "feature"].astype(int).tolist())
print(f"\nF_C = {F_C}  ({len(F_C)} concept-related features)")
cand[["feature", "ratio_abs", "is_member", "confidence", "selected", "description"]]

---- Stage 2 prompt ----
You get a concept name and a feature's description.
Decide if this feature describes the given concept. Consider if the feature is DISTINCTIVE for the concept (not broad like "sport" for "basketball") and not too noisy.
A feature should be marked true only if the concept is central and dominant in the description, not just one of several themes.
Return ONLY JSON: {"is_member": true|false, "confidence": 0..1}

Example 1:
Concept: Charity
Description: This vector represents elements related to giving and donations to charitable causes.
Answer: {"is_member": true, "confidence": 0.98}

Example 2:
Concept: Charity
Description: This vector represents elements related to money, such as gambling, investing and charity.
Answer: {"is_member": false, "confidence": 0.80}

Concept: Ancient Rome
Description: {description}
Answer:


Stage 2: classify: 100%|██████████| 25/25 [00:26<00:00,  1.07s/it]


F_C = [97]  (1 concept-related features)


,feature,ratio_abs,is_member,confidence,selected,description
0,84,6.314508,False,0.85,False,"This feature vector represents military conquest, invasion, and hostile territorial occupation."
1,21,6.099844,False,0.99,False,This vector represents numeric values typically associated with area codes or measurement quantities.
2,22,5.401013,False,0.90,False,"This vector represents historical conflict, governance, and societal upheaval."
3,43,5.263011,False,0.95,False,"This vector represents armed forces and military personnel, equipment, and operations."
4,96,5.027533,False,1.00,False,This feature vector activates on subordinating and coordinating conjunctions or transitional adverbs used to introduce clauses or establish logical relationships.
5,40,4.966961,False,0.99,False,"This feature represents action verbs associated with acquiring, maintaining, or carrying out processes and achievements."
6,55,4.623705,False,1.00,False,These tokens represent indefinite quantities or plural amounts.
7,19,4.482253,False,0.95,False,"This vector represents human societal institutions, belief systems, and cultural norms."
8,23,4.349009,False,0.99,False,"This vector represents the concept of fame, high status, and widespread public acclaim."
9,29,4.149677,False,0.95,False,"These tokens represent a feature vector associated with high social status, recognition, and distinction."


In [46]:
is_true = cand["is_member"] == True
ordered = cand.assign(
    _group=(~is_true).astype(int),
    _key=np.where(is_true, -cand["confidence"], cand["confidence"]),
).sort_values(["_group", "_key"])

for _, row in ordered.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    is_member, conf = verdicts.get(feat, (None, None))
    verdict = "<not classified yet>" if feat not in verdicts else (
        f"is_member={is_member}  confidence={conf if conf is not None else 'n/a'}  "
        f"-> {'SELECTED' if row['selected'] else 'rejected'}"
    )

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {descriptions.get(feat, '<not described yet>')}")
    print(f"  verdict : {verdict}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

=== feature 97  rho=2.04 ===
  Gemini  : This feature vector activates strongly on classical Latin and Roman personal names or name prefixes.
  verdict : is_member=True  confidence=0.95  -> SELECTED
  tokens  : 'Marcus'[C] 0.29, 'Marc'[N] 0.27, 'ĠMarcus'[C] 0.26, 'Mart'[C] 0.26, 'ĠMart'[B] 0.25, 'Luc'[C] 0.24, 'Mar'[N] 0.22, 'ĠLuc'[C] 0.20, 'ĠMar'[N] 0.18, 'Cla'[C] 0.15

=== feature 84  rho=6.31 ===
  Gemini  : This feature vector represents military conquest, invasion, and hostile territorial occupation.
  verdict : is_member=False  confidence=0.85  -> rejected
  tokens  : 'Ġinvading'[C] 0.44, 'Ġinvaded'[C] 0.44, 'Ġinvasion'[C] 0.40, 'Ġinvaders'[C] 0.36, 'Ġsiege'[C] 0.16, 'Ġconquered'[C] 0.16, 'asions'[C] 0.15, 'Ġinv'[C] 0.13, 'Ġoccupying'[N] 0.12, 'Ġinception'[N] 0.12

=== feature 18  rho=2.54 ===
  Gemini  : This feature vector represents the concept of monarchy and imperial rule, encompassing royalty, empires, and related governance terms.
  verdict : is_member=False  confidence=0.

In [47]:
members = cand[cand["is_member"] == True].sort_values("ratio_abs", ascending=False)
print(f"{len(members)} / {len(cand)} candidates have is_member=True\n")

for _, row in members.iterrows():
    feat = int(row["feature"])
    mags = Y[feat].abs()
    top = torch.argsort(mags, descending=True)[:N_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])

    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f} ===")
    print(f"  Gemini  : {row['description']}")
    print(f"  verdict : confidence={row['confidence']:.2f}  -> {'SELECTED' if row['selected'] else 'rejected (confidence < ' + str(CONFIDENCE) + ')'}")
    print("  tokens  : " + ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, top)))
    print()

1 / 35 candidates have is_member=True

=== feature 97  rho=2.04 ===
  Gemini  : This feature vector activates strongly on classical Latin and Roman personal names or name prefixes.
  verdict : confidence=0.95  -> SELECTED
  tokens  : 'Marcus'[C] 0.29, 'Marc'[N] 0.27, 'ĠMarcus'[C] 0.26, 'Mart'[C] 0.26, 'ĠMart'[B] 0.25, 'Luc'[C] 0.24, 'Mar'[N] 0.22, 'ĠLuc'[C] 0.20, 'ĠMar'[N] 0.18, 'Cla'[C] 0.15



### 3.2.f Inspect the selected features

Table 3 of the paper reports only 1–3 embedding features per concept (Harry Potter on Gemma: 1).
Each selected feature should be a tight cluster of concept-specific tokens.

In [48]:
if not F_C:
    print("No feature passed both filters: Step 3.3 will be a no-op for this concept.")

for feat in F_C:
    row = cand.loc[cand["feature"] == feat].iloc[0]
    print(f"=== feature {feat}  rho={row['ratio_abs']:.2f}  conf={row['confidence']:.2f} ===")
    print(f"  {row['description']}")
    print("  tokens:", ", ".join(f"{t!r}[{l[0]}]" for t, l in zip(row["activating_tokens"][:30], row["labels"][:30])))
    print()

=== feature 97  rho=2.04  conf=0.95 ===
  This feature vector activates strongly on classical Latin and Roman personal names or name prefixes.
  tokens: 'Marcus'[A], 'Marc'[N], ' Marcus'[A], 'Mart'[A], ' Mart'[b], 'Luc'[A], 'Mar'[N], ' Luc'[A], ' Mar'[N], 'Cla'[A], ' mar'[A], ' Mari'[b], 'Car'[A], ' Marg'[N], ' Claud'[A], 'Rom'[A], 'Latin'[A], 'Mark'[N], ' Adri'[N], 'Mer'[N], ' Marr'[N], ' marble'[A], 'Carol'[N], ' martial'[A], 'Mac'[A], 'arc'[N], ' Anton'[A], ' Rom'[A], ' Car'[A], 'Roman'[A]



In [49]:
# One column per selected feature: every token with nonzero Y_{i,t}, sorted by |Y_{i,t}|.
columns = {}
for feat in F_C:
    mags = Y[feat].abs()
    nz_idx = torch.nonzero(mags > 0).flatten()
    nz_idx = nz_idx[torch.argsort(mags[nz_idx], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in nz_idx])
    columns[f"feature {feat}"] = pd.Series(
        [f"{t} [{LABEL_TAG[codes[j]]}] {mags[j]:.2f}" for t, j in zip(toks, nz_idx)]
    )

wide = pd.DataFrame(columns).fillna("")
wide.index = range(1, len(wide) + 1)

all_cols = sorted({j for feat in F_C for j in torch.nonzero(Y[feat].abs() > 0).flatten().tolist()})
print(f"{len(F_C)} concept features, {len(all_cols)} distinct tokens "
      f"(C={sum(codes[j] == 1 for j in all_cols)}, N={sum(codes[j] == 0 for j in all_cols)}, "
      f"B={sum(codes[j] == 2 for j in all_cols)})")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(wide)

1 concept features, 50 distinct tokens (C=26, N=22, B=2)


,feature 97
1,Marcus [C] 0.29
2,Marc [N] 0.27
3,ĠMarcus [C] 0.26
4,Mart [C] 0.26
5,ĠMart [B] 0.25
6,Luc [C] 0.24
7,Mar [N] 0.22
8,ĠLuc [C] 0.20
9,ĠMar [N] 0.18
10,Cla [C] 0.15


In [50]:
TOP_N = 30
E_ref = (E_vstar_tl if "E_vstar_tl" in globals() else E_vstar).float().cpu()  # E_vstar is deleted in 3.3.a
token_strs = tokenizer.convert_ids_to_tokens(vstar_ids)

group_masks = {
    "C": codes == 1,
    "C / B": (codes == 1) | (codes == 2),
    "C / B / N": np.ones_like(codes, dtype=bool),
}

for feat in F_C:
    y = Y[feat]
    # z's sign is arbitrary (Y can absorb it); orient it so its support tokens mostly have positive Y.
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots = E_ref @ z
    cos = dots / (E_ref.norm(dim=1) * z.norm())

    df = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot": dots.numpy(),
        "cos": cos.numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df[mask].sort_values("dot", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t, z> --")
        display(top)

=== feature 97   ||z|| = 1.259   sign flipped: True ===

-- C: 1746 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,Luc,C,0.407544,0.460660,0.242310,True
2,Marcus,C,0.407011,0.454371,0.294037,True
3,Mart,C,0.387407,0.454344,0.255158,True
4,ĠMarcus,C,0.377042,0.414634,0.261565,True
5,ĠLuc,C,0.376960,0.411471,0.200423,True
6,Cla,C,0.275836,0.307792,0.154452,True
7,ĠClaud,C,0.269154,0.282029,0.141584,True
8,Ġmar,C,0.265754,0.325391,0.154156,True
9,Car,C,0.262090,0.321855,0.147764,True
10,Rom,C,0.253596,0.295407,0.138088,True



-- C / B: 2494 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,Luc,C,0.407544,0.460660,0.242310,True
2,Marcus,C,0.407011,0.454371,0.294037,True
3,ĠMart,B,0.400097,0.461959,0.252980,True
4,Mart,C,0.387407,0.454344,0.255158,True
5,ĠMarcus,C,0.377042,0.414634,0.261565,True
6,ĠLuc,C,0.376960,0.411471,0.200423,True
7,ĠMari,B,0.291179,0.320153,0.150306,True
8,Cla,C,0.275836,0.307792,0.154452,True
9,ĠClaud,C,0.269154,0.282029,0.141584,True
10,Ġmar,C,0.265754,0.325391,0.154156,True



-- C / B / N: 4973 tokens, top 30 by <e_t, z> --


,token,label,dot,cos,Y,in_support
1,Marc,N,0.437069,0.475325,0.273938,True
2,Luc,C,0.407544,0.460660,0.242310,True
3,Marcus,C,0.407011,0.454371,0.294037,True
4,ĠMart,B,0.400097,0.461959,0.252980,True
5,Mart,C,0.387407,0.454344,0.255158,True
6,Mar,N,0.377291,0.458253,0.224085,True
7,ĠMarcus,C,0.377042,0.414634,0.261565,True
8,ĠLuc,C,0.376960,0.411471,0.200423,True
9,ĠMar,N,0.331259,0.424233,0.182588,True
10,ĠMarg,N,0.296317,0.329457,0.145867,True


In [51]:
E_unit = E_ref / E_ref.norm(dim=1, keepdim=True)

for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    dots_unit = E_unit @ z

    df_unit = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "dot_unit": dots_unit.numpy(),
        "||e_t||": E_ref.norm(dim=1).numpy(),
        "Y": (sign * y).numpy(),
        "in_support": (y != 0).numpy(),
    })

    print(f"=== feature {feat}   ||z|| = {z.norm():.3f}   sign flipped: {sign < 0} ===")
    for name, mask in group_masks.items():
        top = df_unit[mask].sort_values("dot_unit", ascending=False).head(TOP_N).reset_index(drop=True)
        top.index += 1
        print(f"\n-- {name}: {int(mask.sum())} tokens, top {TOP_N} by <e_t / ||e_t||, z> --")
        display(top)

=== feature 97   ||z|| = 1.259   sign flipped: True ===

-- C: 1746 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,Luc,C,0.579952,0.702719,0.242310,True
2,Marcus,C,0.572034,0.711515,0.294037,True
3,Mart,C,0.572001,0.677283,0.255158,True
4,ĠMarcus,C,0.522007,0.722292,0.261565,True
5,ĠLuc,C,0.518025,0.727686,0.200423,True
6,Ġmar,C,0.409653,0.648730,0.154156,True
7,Car,C,0.405202,0.646814,0.147764,True
8,Cla,C,0.387498,0.711839,0.154452,True
9,Rom,C,0.371906,0.681884,0.138088,True
10,ĠClaud,C,0.355063,0.758045,0.141584,True



-- C / B: 2494 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,ĠMart,B,0.581587,0.687940,0.252980,True
2,Luc,C,0.579952,0.702719,0.242310,True
3,Marcus,C,0.572034,0.711515,0.294037,True
4,Mart,C,0.572001,0.677283,0.255158,True
5,ĠMarcus,C,0.522007,0.722292,0.261565,True
6,ĠLuc,C,0.518025,0.727686,0.200423,True
7,Ġmar,C,0.409653,0.648730,0.154156,True
8,Car,C,0.405202,0.646814,0.147764,True
9,ĠMari,B,0.403060,0.722422,0.150306,True
10,Cla,C,0.387498,0.711839,0.154452,True



-- C / B / N: 4973 tokens, top 30 by <e_t / ||e_t||, z> --


,token,label,dot_unit,||e_t||,Y,in_support
1,Marc,N,0.598414,0.730379,0.273938,True
2,ĠMart,B,0.581587,0.687940,0.252980,True
3,Luc,C,0.579952,0.702719,0.242310,True
4,Mar,N,0.576922,0.653972,0.224085,True
5,Marcus,C,0.572034,0.711515,0.294037,True
6,Mart,C,0.572001,0.677283,0.255158,True
7,ĠMar,N,0.534092,0.620229,0.182588,True
8,ĠMarcus,C,0.522007,0.722292,0.261565,True
9,ĠLuc,C,0.518025,0.727686,0.200423,True
10,ĠMarg,N,0.414773,0.714407,0.145867,True


In [52]:
TOP_K_COUNT = 100

rows = []
for feat in F_C:
    y = Y[feat]
    sign = 1.0 if y[y != 0].sum() >= 0 else -1.0
    z = sign * Z[:, feat]
    rankings = {
        "<e_t, z>": E_ref @ z,
        "<e_t / ||e_t||, z>": E_unit @ z,
    }
    for name, scores in rankings.items():
        top = torch.argsort(scores, descending=True)[:TOP_K_COUNT].numpy()
        rows.append({
            "feature": feat,
            "ranking": name,
            "C": int((codes[top] == 1).sum()),
            "B": int((codes[top] == 2).sum()),
            "N": int((codes[top] == 0).sum()),
            "C in support": int(((codes[top] == 1) & (y[top] != 0).numpy()).sum()),
        })

print(f"Label counts among the top {TOP_K_COUNT} of all {len(vstar_ids)} tokens in V* "
      f"(base rate: C={(codes == 1).mean():.1%}, B={(codes == 2).mean():.1%}, N={(codes == 0).mean():.1%})")
pd.DataFrame(rows)

Label counts among the top 100 of all 4973 tokens in V* (base rate: C=35.1%, B=15.0%, N=49.8%)


,feature,ranking,C,B,N,C in support
0,97,"<e_t, z>",40,7,53,25
1,97,"<e_t / ||e_t||, z>",40,8,52,25


In [53]:
# ---- Extended E*: keep Z from 3.1 fixed, refit only a sparse Y' on a larger token set ----
# S_C^ext = every sentence of the live "Ancient Rome" article; S_N^ext = the same number of neutral sentences,
# at most one per page of the local wikimedia/wikipedia sample. Y' is the Y-step of SparseMatrixFactorization.fit
# with F_ = Z frozen:  Y' = WTA_s( (Z^T Z + lambda I)^{-1} Z^T A_ext ),  WTA keeping ceil(s |V*_ext|) tokens per feature.
# With Z fixed this step does not depend on the previous Y', so one pass is already the fixed point of the loop.
import random
import sys

from ember.utils import wta_cols

sys.path.insert(0, "experiments/resample_sentences")
from fetch_en_sentences import neutral_sentences, page_paragraphs, split_sentences

RUN_JOINT_REFIT = False   # True also refits Z and Y jointly on A_ext, to compare time and error

# 1) sentences
canon, revid, paras = page_paragraphs(CONCEPT)
ext_concept = split_sentences(paras)
ext_neutral = [r["sentence"] for r in neutral_sentences(
    RESAMPLED / "wiki_dump_pages.jsonl", len(ext_concept), random.Random(SAMPLE_SEED), exclude=set(ext_concept))]
print(f"S_C^ext = {len(ext_concept)} sentences ('{canon}' rev {revid})   S_N^ext = {len(ext_neutral)} sentences")

# 2) extended token set V*_ext, labelled like build_token_label_codes (1 = concept, 0 = neutral, 2 = both)
special_ids = {tokenizer.pad_token_id, tokenizer.bos_token_id, tokenizer.eos_token_id}
def token_set(sents):
    return {t for s in sents for t in tokenizer(s, add_special_tokens=False)["input_ids"]} - special_ids
c_ids, n_ids = token_set(ext_concept), token_set(ext_neutral)
vstar_ext = sorted(c_ids | n_ids)
codes_ext = np.array([2 if t in c_ids and t in n_ids else 1 if t in c_ids else 0 for t in vstar_ext])
new_tokens = set(vstar_ext) - set(vstar_ids)
print(f"|V*_ext| = {len(vstar_ext)}  (|V*| = {len(vstar_ids)}, {len(new_tokens)} new, "
      f"{len(set(vstar_ids) - set(vstar_ext))} of V* not in V*_ext)")

# 3) A_ext = E_{V*_ext}^T in the factorization space (TL rows; after 3.3.a, HF rows x EMB_SCALE match them)
ids = torch.tensor(vstar_ext)
if "E" in globals():
    E_ext = E.index_select(0, ids.to(E.device)).detach().float().cpu()
else:
    E_ext = W_orig.index_select(0, ids.to(W_orig.device)).detach().float().cpu() * EMB_SCALE
A_ext = E_ext.T.contiguous().to(DEVICE)                                  # (d, |V*_ext|)

# 4) Y-step with Z frozen (same ops as SparseMatrixFactorization.fit, incl. ridge and WTA)
Z_dev = Z.to(DEVICE)
with Timer() as t_ext:
    inv_Z = torch.linalg.inv(Z_dev.T @ Z_dev + torch.eye(Z_dev.shape[1], device=DEVICE) * ridge_used)
    G_ext = A_ext.T @ Z_dev @ inv_Z                                       # (|V*_ext|, k), the G_ layout
    wta_cols(G_ext, pct_keep=SPARSITY)
    if DEVICE != "cpu":
        torch.cuda.synchronize()
Y_ext = G_ext.T.float().cpu()                                             # (k, |V*_ext|)
A_ext_cpu = A_ext.cpu()
ext_err = (torch.norm(A_ext_cpu - Z @ Y_ext) / torch.norm(A_ext_cpu)).item()
print(f"\nfixed-Z fit: {t_ext['elapsed']:.3f}s (joint fit in 3.1.d: {t_fit['elapsed']:.1f}s)   "
      f"WTA keeps {math.ceil(SPARSITY * len(vstar_ext))} tokens per feature")
print(f"relative error on A_ext: fixed Z = {ext_err:.4f}   (3.1 joint fit on A: {rel_err:.4f})")

# 5) the selected concept features on the extended token set
ext_strs = tokenizer.convert_ids_to_tokens(vstar_ext)
for f in F_C:
    mags = Y_ext[f].abs()
    sup = torch.nonzero(mags > 0).flatten()
    sup = sup[torch.argsort(mags[sup], descending=True)].tolist()
    old_sup = {vstar_ids[j] for j in torch.nonzero(Y[f]).flatten().tolist()}
    kept = sum(vstar_ext[j] in old_sup for j in sup)
    n_c = sum(codes_ext[j] == 1 for j in sup)
    print(f"\n=== feature {f}: {len(sup)} tokens in Y'_ext support | {n_c} concept-only | "
          f"{kept}/{len(old_sup)} of its 3.1 support kept | {sum(vstar_ext[j] in new_tokens for j in sup)} new tokens ===")
    print(", ".join(f"{ext_strs[j]!r}[{LABEL_TAG[codes_ext[j]]}{'*' if vstar_ext[j] in new_tokens else ''}]"
                    for j in sup))
print("\n(* = token not in V*)")

# 6) optional: joint Z, Y refit on A_ext with the 3.1 settings, for the speed / error comparison
if RUN_JOINT_REFIT:
    set_seed(SEED)
    smf_ext = SparseMatrixFactorization(RANK, fitting_device=DEVICE, g_sparsity=SPARSITY)
    with Timer() as t_joint:
        fit_with_ridge(smf_ext, A_ext, MAX_ITER, patience=PATIENCE, base_reg=RIDGE)
    joint_err = (torch.norm(A_ext_cpu - smf_ext.F_.detach().cpu() @ smf_ext.G_.detach().cpu().T)
                 / torch.norm(A_ext_cpu)).item()
    print(f"\njoint refit on A_ext: {t_joint['elapsed']:.1f}s, relative error {joint_err:.4f}   "
          f"vs fixed Z: {t_ext['elapsed']:.3f}s, {ext_err:.4f}")


S_C^ext = 710 sentences ('Ancient Rome' rev 1379589331)   S_N^ext = 710 sentences
|V*_ext| = 8474  (|V*| = 4973, 3501 new, 0 of V* not in V*_ext)

fixed-Z fit: 0.012s (joint fit in 3.1.d: 20.0s)   WTA keeps 85 tokens per feature
relative error on A_ext: fixed Z = 0.9632   (3.1 joint fit on A: 0.9508)

=== feature 97: 85 tokens in Y'_ext support | 29 concept-only | 50/50 of its 3.1 support kept | 32 new tokens ===
'Marcus'[C], 'Marc'[N], 'ĠMarcus'[C], 'Mart'[C], 'ĠMart'[B], 'Luc'[B], 'Mar'[N], 'ĠMarc'[C*], 'ĠLuc'[C], 'ĠMar'[B], 'Cla'[C], 'Ġmar'[B], 'ĠMari'[B], 'Car'[C], 'ĠMarg'[N], 'ĠClaud'[C], 'Rom'[C], 'Latin'[C], 'Mark'[N], 'ĠAdri'[N], 'Mer'[N], 'ĠMarr'[N], 'Ġmarble'[C], 'Carol'[N], 'Ġmartial'[C], 'Maria'[N*], 'Jul'[C*], 'Mac'[B], 'arc'[N], 'ĠAnton'[C], 'ĠRom'[C], 'ĠCar'[B], 'Kar'[N*], 'Roman'[C], 'March'[C], 'ĠGiul'[N], 'Sus'[N], 'ĠLatin'[C], 'Ġmarine'[N], 'Mary'[C], 'ĠKar'[N], 'ĠMarie'[N*], 'ĠMartin'[N], 'Pat'[B], 'Med'[C], 'ĠMare'[N*], 'ĠMaria'[N*], 'Mah'[N], 'Carl'[N*], 'ĠJulio'[

In [54]:
# Token counts per group: original V* (3.1) vs extended V*_ext, split into tokens already in V* and new ones (*)
lab_v = {t: LABEL_TAG[int(c)] for t, c in zip(vstar_ids, codes)}
lab_x = {t: LABEL_TAG[int(c)] for t, c in zip(vstar_ext, codes_ext)}
group_counts = pd.DataFrame({
    "V* (original)": pd.Series(lab_v).value_counts(),
    "V*_ext (all)": pd.Series(lab_x).value_counts(),
    "V*_ext, also in V*": pd.Series({t: l for t, l in lab_x.items() if t not in new_tokens}).value_counts(),
    "V*_ext, new (*)": pd.Series({t: l for t, l in lab_x.items() if t in new_tokens}).value_counts(),
}).reindex(["C", "N", "B"]).fillna(0).astype(int)
group_counts.loc["total"] = group_counts.sum()
group_counts["V*_ext / V*"] = (group_counts["V*_ext (all)"] / group_counts["V* (original)"]).round(2)
display(group_counts)

# Tokens in both sets whose label changed (e.g. N in V* becomes B because the token also occurs in a new AR sentence)
both_sets = [t for t in vstar_ids if t in lab_x]
display(pd.crosstab(pd.Series([lab_v[t] for t in both_sets], name="label in V*"),
                    pd.Series([lab_x[t] for t in both_sets], name="label in V*_ext"), margins=True))


,V* (original),V*_ext (all),"V*_ext, also in V*","V*_ext, new (*)",V*_ext / V*
C,1746,2680,1415,1265,1.53
N,2479,4171,2133,2038,1.68
B,748,1623,1425,198,2.17
total,4973,8474,4973,3501,1.70


label in V*_ext,B,C,N,All
label in V*,,,,
B,748,0,0,748
C,331,1415,0,1746
N,346,0,2133,2479
All,1425,1415,2133,4973


In [55]:
# 5) the selected concept features on the extended token set, ranked by |Y'| (signed value shown)
ext_strs = tokenizer.convert_ids_to_tokens(vstar_ext)
for f in F_C:
    mags = Y_ext[f].abs()
    sup = torch.nonzero(mags > 0).flatten()
    sup = sup[torch.argsort(mags[sup], descending=True)].tolist()
    old_sup = {vstar_ids[j] for j in torch.nonzero(Y[f]).flatten().tolist()}
    print(f"=== feature {f}: {len(sup)} tokens in Y'_ext support | {sum(codes_ext[j] == 1 for j in sup)} concept-only | "
          f"{sum(vstar_ext[j] in old_sup for j in sup)}/{len(old_sup)} of its 3.1 support kept | "
          f"{sum(vstar_ext[j] in new_tokens for j in sup)} new tokens (*) ===")
    table = pd.DataFrame({
        "token": [f"{ext_strs[j]!r} [{LABEL_TAG[codes_ext[j]]}{'*' if vstar_ext[j] in new_tokens else ''}]" for j in sup],
        f"Y'[{f}]": [Y_ext[f, j].item() for j in sup],
        "in 3.1 support": [vstar_ext[j] in old_sup for j in sup],
    })
    table.index = pd.RangeIndex(1, len(table) + 1, name="rank")
    display(table.style.format({f"Y'[{f}]": "{:+.4f}"}))

=== feature 97: 85 tokens in Y'_ext support | 29 concept-only | 50/50 of its 3.1 support kept | 32 new tokens (*) ===


,token,Y'[97],in 3.1 support
rank,,,
1,'Marcus' [C],-0.2940,True
2,'Marc' [N],-0.2739,True
3,'ĠMarcus' [C],-0.2616,True
4,'Mart' [C],-0.2552,True
5,'ĠMart' [B],-0.2530,True
6,'Luc' [B],-0.2423,True
7,'Mar' [N],-0.2241,True
8,'ĠMarc' [C*],-0.2046,False
9,'ĠLuc' [C],-0.2004,True


In [56]:
# ---- Dense refit with unit-norm directions: fix Zn, solve Y' = argmin ||E*^T - Zn Y'||_F (no sparsity) ----
# Scale convention for this analysis: z_i <- z_i / ||z_i||  and  Y_i <- ||z_i|| * Y_i, so Z Y is unchanged and
# Y' values are comparable across directions. Z and Y stay as fitted (Step 3.3 uses them); the copies are Zn, Yn.
TOP_DIRS = 5
AR_FEATS = list(F_C)          # the Ancient Rome concept column(s) of Z; override e.g. [37] if you want one
AR_MARK = "★★★ AR ★★★"

z_scale = Z.norm(dim=0)                    # (k,)  ||z_i||: the scale moved from Z into Y
Zn = Z / z_scale                           # (d, k)       unit-norm directions
Yn = Y * z_scale[:, None]                  # (k, |V*|)    sparse Y, rows scaled by ||z_i||
k_all = len(z_scale)

A_ref = (E_vstar_tl if "E_vstar_tl" in globals() else E_vstar).double().cpu().T   # (d, |V*|)
Zn64 = Zn.double()                                                                  # (d, k)
Y_dense = torch.linalg.lstsq(Zn64, A_ref).solution.float()                          # (k, |V*|), unit-norm scale

dense_err = (torch.norm(A_ref - Zn64 @ Y_dense.double()) / torch.norm(A_ref)).item()
sparse_err = (torch.norm(A_ref - Zn64 @ Yn.double()) / torch.norm(A_ref)).item()
print(f"relative error: sparse Yn = {sparse_err:.4f}   dense Y' = {dense_err:.4f}")

# Gemini descriptions for every direction. Only missing ones are computed; existing ones are kept.
# Kept apart from `descriptions` so that re-running Stage 2 (3.2.e) still judges only the tau-passing candidates.
all_descriptions = globals().get("all_descriptions", {})
all_descriptions.update({f: d for f, d in descriptions.items() if d})   # reuse Step 3.2's descriptions
todo = [f for f in range(k_all) if not all_descriptions.get(f)]
print(f"{k_all - len(todo)} directions already described, {len(todo)} to compute")
for f in tqdm(todo, desc="describe"):
    all_descriptions[f] = get_description(client, feat_tokens.loc[f, "activating_tokens"][:TOP_K_TOKENS])

# Scale of every direction z_i (largest first), with its description
scale_df = pd.DataFrame({"||z_i||": z_scale.numpy(), "||Y_i|| (as fitted)": Y.norm(dim=1).numpy(),
                         "||Yn_i||": Yn.norm(dim=1).numpy(), "in F_C": [f in AR_FEATS for f in range(k_all)],
                         "description": [all_descriptions.get(f, "") for f in range(k_all)]})
scale_df.index.name = "feature"
scale_df = scale_df.sort_values("||z_i||", ascending=False)
scale_df.insert(0, "rank by ||z||", range(1, len(scale_df) + 1))
print(f"||z_i|| over all {k_all} directions: min = {z_scale.min():.4f}  median = {z_scale.median():.4f}  "
      f"max = {z_scale.max():.4f}")
for f in AR_FEATS:
    print(f"AR feature {f}: ||z|| = {z_scale[f]:.4f}  (rank {int(scale_df.loc[f, 'rank by ||z||'])}/{k_all})")
display(scale_df.style.format({"||z_i||": "{:.4f}", "||Y_i|| (as fitted)": "{:.4f}", "||Yn_i||": "{:.4f}"})
        .set_properties(subset=["description"], **{"text-align": "left", "white-space": "pre-wrap"}))

for ar in AR_FEATS:
    mags = Yn[ar].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in support])

    print(f"\n{'=' * 100}\n=== AR feature {ar} (||z|| = {z_scale[ar]:.4f}): {len(support)} tokens with nonzero Y, "
          f"by decreasing |Yn| ===\n{'=' * 100}")
    for rank_t, (j, tok) in enumerate(zip(support, toks), 1):
        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        top = order[:TOP_DIRS]

        print(f"\n{rank_t:3d}. {tok!r} [{LABEL_TAG[codes[j]]}]   sparse Yn[{ar}] = {Yn[ar, j]:+.3f}")
        for r, f in enumerate(top, 1):
            mark = f"   <-- {AR_MARK}" if f in AR_FEATS else ""
            print(f"       #{r}  z_{f:<4d} ||z|| = {z_scale[f]:.3f}   Y' = {y_dense_t[f]:+.3f}{mark}")
        for f in AR_FEATS:
            if f not in top:
                print(f"       ..  z_{f:<4d} ||z|| = {z_scale[f]:.3f}   Y' = {y_dense_t[f]:+.3f}   <-- {AR_MARK} "
                      f"(rank {order.index(f) + 1}/{len(order)})")


relative error: sparse Yn = 0.9510   dense Y' = 0.9210
96 directions already described, 4 to compute


describe: 100%|██████████| 4/4 [00:05<00:00,  1.33s/it]

||z_i|| over all 100 directions: min = 0.9931  median = 1.3451  max = 1.9047
AR feature 97: ||z|| = 1.2590  (rank 68/100)


,rank by ||z||,||z_i||,||Y_i|| (as fitted),||Yn_i||,in F_C,description
feature,,,,,,
7,1,1.9047,1.0001,1.9048,False,"This feature vector represents concepts related to society, community, civic life, and social relations."
28,2,1.8807,1.0001,1.8808,False,"This feature vector represents the concept of winning, success, and competitive outcomes or achievements."
44,3,1.8497,1.0001,1.8498,False,This feature vector represents the letter 'z' and names or words containing the letter 'z'.
24,4,1.8403,1.0000,1.8404,False,"This vector activates strongly for words containing the letter combination ""al"" or related linguistic roots."
73,5,1.8211,1.0001,1.8212,False,"This feature activates on words and proper nouns starting with the letters ""Tr"" or ""T""."
3,6,1.7749,1.0000,1.7750,False,"This feature activates for words and morphemes related to historical resistance movements, rebellion, and political conflict."
77,7,1.7420,1.0000,1.7421,False,This vector represents common alphanumeric character sequences and prefixes beginning with the letter 'r'.
95,8,1.7033,1.0001,1.7034,False,"This vector represents single-letter alphabetic tokens, commonly used as text abbreviations, variables, or categorical markers."
21,9,1.6970,1.0000,1.6971,False,This vector represents numeric values typically associated with area codes or measurement quantities.



=== AR feature 97 (||z|| = 1.2590): 50 tokens with nonzero Y, by decreasing |Yn| ===

  1. 'Marcus' [C]   sparse Yn[97] = -0.370
       #1  z_97   ||z|| = 1.259   Y' = -0.370   <-- ★★★ AR ★★★
       #2  z_16   ||z|| = 1.031   Y' = -0.253
       #3  z_5    ||z|| = 1.292   Y' = -0.068
       #4  z_39   ||z|| = 1.428   Y' = -0.055
       #5  z_81   ||z|| = 1.281   Y' = -0.051

  2. 'Marc' [N]   sparse Yn[97] = -0.345
       #1  z_97   ||z|| = 1.259   Y' = -0.345   <-- ★★★ AR ★★★
       #2  z_82   ||z|| = 1.443   Y' = -0.057
       #3  z_77   ||z|| = 1.742   Y' = +0.034
       #4  z_5    ||z|| = 1.292   Y' = -0.028
       #5  z_41   ||z|| = 1.338   Y' = +0.025

  3. 'ĠMarcus' [C]   sparse Yn[97] = -0.329
       #1  z_97   ||z|| = 1.259   Y' = -0.329   <-- ★★★ AR ★★★
       #2  z_16   ||z|| = 1.031   Y' = -0.274
       #3  z_5    ||z|| = 1.292   Y' = -0.079
       #4  z_39   ||z|| = 1.428   Y' = -0.078
       #5  z_81   ||z|| = 1.281   Y' = -0.044

  4. 'Mart' [C]   sparse Yn[97] = -0.321


In [57]:
# ---- All C-labelled tokens, sorted by |Y'_AR| / max_{i != AR} |Y'_i| (unit-norm Z; = top-1/top-2 when AR is top-1) ----
for ar in AR_FEATS:
    c_cols = np.where(codes == 1)[0].tolist()
    rows = []
    for j in c_cols:
        y_dense_t = Y_dense[:, j]
        others_max = torch.cat([y_dense_t[:ar], y_dense_t[ar + 1:]]).abs().max().item()
        rows.append((y_dense_t[ar].abs().item() / others_max, j))
    rows.sort(reverse=True)

    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for _, j in rows])
    n_pass = sum(r >= RATIO for r, _ in rows)
    n_top1 = sum(r >= 1.0 for r, _ in rows)
    print(f"{'=' * 100}\n=== AR feature {ar} (||z|| = {z_scale[ar]:.4f}): {len(rows)} C tokens | AR top-1: {n_top1} | "
          f"AR top-1 and >= {RATIO:g}x: {n_pass} ===\n{'=' * 100}")

    shown_ratio_line = shown_top1_line = False
    for rank_t, ((ratio, j), tok) in enumerate(zip(rows, toks), 1):
        if not shown_ratio_line and ratio < RATIO:
            print(f"\n{'-' * 40} below {RATIO:g}x {'-' * 40}")
            shown_ratio_line = True
        if not shown_top1_line and ratio < 1.0:
            print(f"\n{'-' * 36} AR no longer top-1 {'-' * 36}")
            shown_top1_line = True

        y_dense_t = Y_dense[:, j]
        order = torch.argsort(y_dense_t.abs(), descending=True).tolist()
        in_support = "in AR support" if Yn[ar, j] != 0 else "not in AR support"
        print(f"\n{rank_t:3d}. {tok!r} [C]   ratio = {ratio:.2f}x   sparse Yn[{ar}] = {Yn[ar, j]:+.3f}  ({in_support})")
        for r, f in enumerate(order[:TOP_DIRS], 1):
            mark = f"   <-- {AR_MARK}" if f in AR_FEATS else ""
            print(f"       #{r}  z_{f:<4d} ||z|| = {z_scale[f]:.3f}   Y' = {y_dense_t[f]:+.3f}{mark}")
        if ar not in order[:TOP_DIRS]:
            print(f"       ..  z_{ar:<4d} ||z|| = {z_scale[ar]:.3f}   Y' = {y_dense_t[ar]:+.3f}   <-- {AR_MARK} "
                  f"(rank {order.index(ar) + 1}/{len(order)})")


NameError: name 'RATIO' is not defined

In [ ]:
# ---- Extended E* = Z Y' with unit-norm directions: scale table, AR tokens, and the AR-dominance ratio for every token ----
# Uses the fixed-Z fit on V*_ext (cell "Extended E*": Y_ext, A_ext, vstar_ext, codes_ext, new_tokens, ext_strs).
# New scale: z_i <- z_i / ||z_i||, Y'_i <- ||z_i|| Y'_i (all scale in the rows of Y'); Z Y' is unchanged.
# Old scale = as fitted with the 3.1 Z (column norms ||z_i||), shown next to every value for comparison.
TOP_DIRS = 5
RATIO = 2.0
RATIO_STEPS = [RATIO, 3, 4, 5, 6, 7, 8]   # thresholds counted in the summary table
DETAIL_MIN_RATIO = 1.0        # full 5-direction breakdown for tokens with ratio >= this; the table keeps all tokens
AR_FEATS = list(F_C)
AR_MARK = "★★★ AR ★★★"

z_scale = Z.norm(dim=0)                                         # (k,) same Z as 3.1, so the same ||z_i||
Zn = Z / z_scale
k_all = len(z_scale)
lab_ext = [LABEL_TAG[int(c)] + ("*" if t in new_tokens else "") for c, t in zip(codes_ext, vstar_ext)]
LABELS = ["C", "C*", "N", "N*", "B", "B*"]

# 1) sparse (fixed-Z WTA fit) and dense refits on A_ext, in the new scale
Yn_ext = Y_ext * z_scale[:, None]                               # (k, |V*_ext|) sparse, new scale
A_ext64 = A_ext.double().cpu()                                  # (d, |V*_ext|)
Zn64 = Zn.double()
Yd_ext = torch.linalg.lstsq(Zn64, A_ext64).solution.float()     # (k, |V*_ext|) dense Y', new scale
dense_err_ext = (torch.norm(A_ext64 - Zn64 @ Yd_ext.double()) / torch.norm(A_ext64)).item()
sparse_err_ext = (torch.norm(A_ext64 - Zn64 @ Yn_ext.double()) / torch.norm(A_ext64)).item()
print(f"|V*_ext| = {len(vstar_ext)}   relative error on A_ext: sparse Yn_ext = {sparse_err_ext:.4f}   "
      f"dense Y' = {dense_err_ext:.4f}")

# 2) scale of every direction as a distribution: share of the squared Frobenius norm, ||v_i||^2 / sum_j ||v_j||^2
def norm_share(rows_norm: torch.Tensor) -> torch.Tensor:
    sq = rows_norm.double() ** 2
    return (100 * sq / sq.sum()).float()

z_pct = norm_share(z_scale)                                     # % of ||Z||_F^2 in column z_i
y_old_pct = norm_share(Y_ext.norm(dim=1))                       # % of ||Y'||_F^2 in row i, old scale
y_new_pct = norm_share(Yn_ext.norm(dim=1))                      # % of ||Yn'||_F^2 in row i, new scale

all_descriptions = globals().get("all_descriptions", {})
scale_ext_df = pd.DataFrame({
    "||z_i||": z_scale.numpy(),
    "||z_i|| %": z_pct.numpy(),
    "||Y'_i|| old scale": Y_ext.norm(dim=1).numpy(),
    "||Y'_i|| % old scale": y_old_pct.numpy(),
    "||Y'_i|| new scale": Yn_ext.norm(dim=1).numpy(),
    "||Y'_i|| % new scale": y_new_pct.numpy(),
    "|supp|": (Y_ext != 0).sum(1).numpy(),
    "in F_C": [f in AR_FEATS for f in range(k_all)],
    "description": [all_descriptions.get(f, "") for f in range(k_all)],
})
scale_ext_df.index.name = "feature"
scale_ext_df = scale_ext_df.sort_values("||z_i|| %", ascending=False)
scale_ext_df.insert(0, "rank by ||z||", range(1, k_all + 1))
print(f"||z_i||: min = {z_scale.min():.4f}  median = {z_scale.median():.4f}  max = {z_scale.max():.4f}")
print(f"share of ||Z||_F^2 per direction: min = {z_pct.min():.2f}%  median = {z_pct.median():.2f}%  "
      f"max = {z_pct.max():.2f}%  (uniform = {100 / k_all:.2f}%)")
pct = "{:.2f}%"
display(scale_ext_df.style.format({"||z_i||": "{:.4f}", "||z_i|| %": pct, "||Y'_i|| old scale": "{:.4f}",
                                    "||Y'_i|| % old scale": pct, "||Y'_i|| new scale": "{:.4f}", "||Y'_i|| % new scale": pct})
        .set_properties(subset=["description"], **{"text-align": "left", "white-space": "pre-wrap"}))


def print_dirs(j: int, ar: int):
    """Top-TOP_DIRS directions of token column j in the dense Y' (new scale), with the old-scale value."""
    y = Yd_ext[:, j]
    order = torch.argsort(y.abs(), descending=True).tolist()
    for r, f in enumerate(order[:TOP_DIRS], 1):
        mark = f"   <-- {AR_MARK}" if f in AR_FEATS else ""
        print(f"       #{r}  z_{f:<4d} ||z|| = {z_scale[f]:.3f} ({z_pct[f]:.2f}%)   Y' = {y[f]:+.3f}   (old {y[f] / z_scale[f]:+.4f}){mark}")
    for f in AR_FEATS:
        if f not in order[:TOP_DIRS]:
            print(f"       ..  z_{f:<4d} ||z|| = {z_scale[f]:.3f} ({z_pct[f]:.2f}%)   Y' = {y[f]:+.3f}   (old {y[f] / z_scale[f]:+.4f})"
                  f"   <-- {AR_MARK} (rank {order.index(f) + 1}/{len(order)})")


for ar in AR_FEATS:
    # 3) AR support in the sparse fit, by decreasing |Yn_ext|
    mags = Yn_ext[ar].abs()
    support = torch.nonzero(mags > 0).flatten()
    support = support[torch.argsort(mags[support], descending=True)].tolist()
    counts = "  ".join(f"{l}: {sum(lab_ext[j] == l for j in support)}" for l in LABELS)
    print(f"\n{'=' * 100}\n=== AR feature {ar} (||z|| = {z_scale[ar]:.4f}, {z_pct[ar]:.2f}% of ||Z||_F^2): {len(support)} tokens with nonzero Y' on "
          f"V*_ext, by decreasing |Yn_ext|   [{counts}] ===\n{'=' * 100}")
    for rank_t, j in enumerate(support, 1):
        print(f"\n{rank_t:3d}. {ext_strs[j]!r} [{lab_ext[j]}]   sparse Yn_ext[{ar}] = {Yn_ext[ar, j]:+.3f}   "
              f"(old {Y_ext[ar, j]:+.4f})")
        print_dirs(j, ar)

    # 4) every V*_ext token, sorted by |Y'_AR| / max_{i != AR} |Y'_i| (dense, new scale; = top-1/top-2 when AR is top-1)
    absd = Yd_ext.abs()
    others = absd.clone()
    others[ar] = -1.0
    ratio = (absd[ar] / others.max(0).values).numpy()
    order_t = np.argsort(-ratio)
    in_sup = (Yn_ext[ar] != 0).numpy()
    ratio_ext_df = pd.DataFrame({
        "token": [f"{ext_strs[j]!r} [{lab_ext[j]}]" for j in order_t],
        "label": [lab_ext[j] for j in order_t],
        "ratio": ratio[order_t],
        "Y'_AR new": Yd_ext[ar].numpy()[order_t],
        "Y'_AR old": (Yd_ext[ar] / z_scale[ar]).numpy()[order_t],
        "in AR support": in_sup[order_t],
    }, index=pd.RangeIndex(1, len(order_t) + 1, name="rank"))

    summary = pd.DataFrame({
        "tokens": ratio_ext_df["label"].value_counts(),
        "AR top-1": ratio_ext_df.loc[ratio_ext_df["ratio"] >= 1.0, "label"].value_counts(),
        **{f">= {t:g}x": ratio_ext_df.loc[ratio_ext_df["ratio"] >= t, "label"].value_counts() for t in RATIO_STEPS},
        "in AR support": ratio_ext_df.loc[ratio_ext_df["in AR support"], "label"].value_counts(),
    }).reindex(LABELS).fillna(0).astype(int)
    summary.loc["total"] = summary.sum()
    print(f"\n{'=' * 100}\n=== AR feature {ar}: all {len(ratio_ext_df)} V*_ext tokens by |Y'_AR| / max other |Y'| ===\n"
          f"{'=' * 100}")
    display(summary)

    shown_ratio_line = shown_top1_line = False
    for rank_t, j in enumerate(order_t, 1):
        r_t = ratio[j]
        if r_t < DETAIL_MIN_RATIO:
            print(f"\n... {len(order_t) - rank_t + 1} more tokens with ratio < {DETAIL_MIN_RATIO:g}x: see ratio_ext_df")
            break
        if not shown_ratio_line and r_t < RATIO:
            print(f"\n{'-' * 40} below {RATIO:g}x {'-' * 40}")
            shown_ratio_line = True
        if not shown_top1_line and r_t < 1.0:
            print(f"\n{'-' * 36} AR no longer top-1 {'-' * 36}")
            shown_top1_line = True
        sup_txt = "in AR support" if in_sup[j] else "not in AR support"
        print(f"\n{rank_t:4d}. {ext_strs[j]!r} [{lab_ext[j]}]   ratio = {r_t:.2f}x   "
              f"sparse Yn_ext[{ar}] = {Yn_ext[ar, j]:+.3f} (old {Y_ext[ar, j]:+.4f})  ({sup_txt})")
        print_dirs(j, ar)

    display(ratio_ext_df.head(200).style.format({"ratio": "{:.2f}", "Y'_AR new": "{:+.3f}", "Y'_AR old": "{:+.4f}"}))


|V*_ext| = 8317   relative error on A_ext: sparse Yn_ext = 0.9447   dense Y' = 0.9032
||z_i||: min = 0.9274  median = 1.1869  max = 1.8499
share of ||Z||_F^2 per direction: min = 0.29%  median = 0.47%  max = 1.15%  (uniform = 0.50%)


,rank by ||z||,||z_i||,||z_i|| %,||Y'_i|| old scale,||Y'_i|| % old scale,||Y'_i|| new scale,||Y'_i|| % new scale,|supp|,in F_C,description
feature,,,,,,,,,,
191,1,1.8499,1.15%,1.1120,0.53%,2.0571,1.19%,84,False,"This feature vector is related to sequential three-digit integers, likely representing numerical IDs, years, or index values in a dataset."
86,2,1.8075,1.09%,1.0663,0.49%,1.9273,1.05%,84,False,This vector represents calendar days of the month.
147,3,1.7667,1.05%,1.1211,0.54%,1.9807,1.11%,84,False,The tokens represent the months of the year in various spellings and abbreviations.
182,4,1.6943,0.96%,1.1572,0.57%,1.9606,1.08%,84,False,This vector represents nationalities and languages associated with various countries around the world.
149,5,1.6701,0.93%,1.2367,0.65%,2.0654,1.20%,84,False,This vector represents common masculine given names in English.
50,6,1.6489,0.91%,1.1083,0.52%,1.8275,0.94%,84,False,"Adverbs of time, frequency, and manner describing how or when actions occur."
135,7,1.6372,0.90%,1.1668,0.58%,1.9104,1.03%,84,False,"This vector represents two-digit numbers, particularly within the range of the fifties through the nineties."
136,8,1.6343,0.90%,1.1216,0.54%,1.8331,0.95%,84,False,These tokens represent uppercase consonant letters of the English alphabet.
111,9,1.5640,0.82%,1.0884,0.51%,1.7022,0.82%,84,False,This vector represents cardinal directions and their directional derivatives.


In [ ]:
# ---- Geometry of each AR direction in Z: angle to the span of the other columns, closest and most orthogonal z_i ----
# All of it is scale-free (angles, cosines, supports), so unit-norm Zn gives the same numbers as Z.
N_DIRS = 5            # closest / most orthogonal directions shown per AR feature
N_TOKENS_SHOW = 15

Zn64 = Zn.double()
d, k = Zn64.shape
all_descriptions = globals().get("all_descriptions", {})


def desc(f: int) -> str:
    """Gemini description of z_f; computed only if missing, stored in all_descriptions (not in Step 3.2's dict)."""
    if not all_descriptions.get(f):
        all_descriptions[f] = descriptions.get(f) or get_description(
            client, feat_tokens.loc[f, "activating_tokens"][:TOP_K_TOKENS])
    return all_descriptions[f]


def top_tokens(f: int) -> str:
    mags = Yn[f].abs()
    top = torch.argsort(mags, descending=True)[:N_TOKENS_SHOW]
    top = top[mags[top] > 0].tolist()
    toks = tokenizer.convert_ids_to_tokens([vstar_ids[j] for j in top])
    return ", ".join(f"{t!r}[{LABEL_TAG[codes[j]]}]" for t, j in zip(toks, top))


def angle_to_span(f: int, exclude=()):
    """Angle (deg) between unit z_f and span{z_i : i != f, i not in exclude}, plus the least-squares coefficients."""
    others = [i for i in range(k) if i != f and i not in exclude]
    coef = torch.linalg.lstsq(Zn64[:, others], Zn64[:, [f]]).solution.squeeze(1)
    sin = (Zn64[:, f] - Zn64[:, others] @ coef).norm().item()      # ||z_f|| = 1, so the residual norm is sin
    return math.degrees(math.asin(min(sin, 1.0))), others, coef


all_angles = pd.Series({f: angle_to_span(f)[0] for f in range(k)}).sort_values()
rand_angle = math.degrees(math.acos(math.sqrt((k - 1) / d)))
print(f"d = {d}, k = {k}. Angle of each z_i to the span of the other {k - 1}: "
      f"median {all_angles.median():.2f}°, range {all_angles.min():.2f}°-{all_angles.max():.2f}° "
      f"(a random direction vs a random {k - 1}-dim subspace: ~{rand_angle:.1f}°)")

cos_mat = Zn64.T @ Zn64                                             # (k, k) cosines between directions
for ar in AR_FEATS:
    angle, others, coef = angle_to_span(ar)
    top3 = ", ".join(f"z_{others[i]}: {coef[i]:+.2f}" for i in torch.argsort(coef.abs(), descending=True)[:3].tolist())
    print(f"\n{'=' * 100}\n=== AR feature {ar}   ||z|| = {z_scale[ar]:.4f} ===\n{'=' * 100}")
    print(f"  Gemini : {desc(ar)}\n  tokens : {top_tokens(ar)}")
    print(f"  angle to span of the others = {angle:.2f}°  -> more orthogonal than "
          f"{(all_angles < all_angles[ar]).mean():.0%} of directions (rank {all_angles.index.get_loc(ar) + 1}/{k}, "
          f"ascending)   top-3 lstsq coef: {top3}")
    if len(AR_FEATS) > 1:
        print(f"  angle to span of the non-AR directions only = {angle_to_span(ar, exclude=AR_FEATS)[0]:.2f}°")

    cos_ar = cos_mat[ar].clone()
    cos_ar[ar] = float("nan")
    by_abs = cos_ar.abs().nan_to_num(-1.0)
    closest = torch.argsort(by_abs, descending=True)[:N_DIRS].tolist()
    ortho = torch.argsort(cos_ar.abs().nan_to_num(2.0))[:N_DIRS].tolist()
    rows = []
    for group, feats in [("closest", closest), ("most orthogonal", ortho)]:
        for r, f in enumerate(feats, 1):
            c = cos_ar[f].item()
            rows.append({"group": group, "rank": r, "feature": f, "||z||": z_scale[f].item(),
                         "angle_deg": math.degrees(math.acos(min(abs(c), 1.0))), "cos (signed)": c,
                         "shared tokens": int(((Yn[ar] != 0) & (Yn[f] != 0)).sum()),
                         "description": desc(f), "top tokens": top_tokens(f)})
    display(pd.DataFrame(rows).set_index(["group", "rank"]).style
            .format({"||z||": "{:.4f}", "angle_deg": "{:.2f}", "cos (signed)": "{:+.4f}"})
            .set_properties(subset=["description", "top tokens"], **{"text-align": "left", "white-space": "pre-wrap"}))


d = 4096, k = 200. Angle of each z_i to the span of the other 199: median 58.43°, range 41.84°-71.59° (a random direction vs a random 199-dim subspace: ~77.3°)


---
# Step 3.3: Erasing concept-related features

The factorization writes each embedding as a sum of feature contributions plus residual error:

$$\mathbf e_t = \underbrace{\sum_{i\in F_C} Y_{i,t}\mathbf z_i}_{\mathbf c_t\ \text{(concept part)}} + \sum_{i\notin F_C} Y_{i,t}\mathbf z_i + \boldsymbol\varepsilon_t
\quad\Longrightarrow\quad \mathbf e_t' = \mathbf e_t - \delta\,\mathbf c_t$$

The edit applies **only** to concept-only tokens ($t \in T_C$) that have $Y_{i,t}\neq 0$ for some $i\in F_C$.
All other rows of $E$ stay unchanged, and so does the unembedding $U$. With $\delta=1$ the edit removes exactly
the recovered concept component. With $\delta>1$ it over-subtracts, which the paper finds often erases better.

This step keeps **two models** side by side: `original_model` (untouched) and `erased_model`
(the same weights with the EMBER edit applied), plus a small API to query and compare them.


### 3.3.a Swap the TL model for the original HF model

Erasure and evaluation run on HF models. Llama-3.1-8B's `lm_head` is already separate from the input
embeddings, so editing $E$ leaves $U$ unchanged, as the paper does. Then a sanity check confirms that
HF `W_E` × scale matches the TL rows the model factorized. That match is what makes the write-back in 3.3.c correct.

In [ ]:
# Re-run safe: the TL objects are freed on the first run, and original_model is never modified.
if "E_vstar" in globals():
    E_vstar_tl = E_vstar.cpu()  # factorization-space reference rows
    del act_generator, tl_model, E, E_vstar, A, A_cpu, smf
    gc.collect()
    torch.cuda.empty_cache()

if "original_model" not in globals():
    original_model, hf_tok = load_hf_model(MODEL_NAME)
    original_model.eval()
    original_model.config.use_cache = True  # loader disables the KV cache for training; we only generate

W_orig = original_model.get_input_embeddings().weight
EMB_SCALE = math.sqrt(W_orig.shape[1]) if "gemma" in MODEL_NAME.lower() else 1.0

with torch.no_grad():
    hf_rows = W_orig[torch.tensor(vstar_ids, device=W_orig.device)].float().cpu() * EMB_SCALE
cos = F.cosine_similarity(hf_rows, E_vstar_tl, dim=1)
norm_ratio = hf_rows.norm(dim=1) / E_vstar_tl.norm(dim=1)
print(f"EMB_SCALE={EMB_SCALE:.3f} | cos(HF*scale, TL): min={cos.min():.4f} | norm ratio: "
      f"[{norm_ratio.min():.3f}, {norm_ratio.max():.3f}]")
assert cos.min() > 0.99 and (norm_ratio - 1).abs().max() < 0.02, "HF and TL embedding spaces do not match"


Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 70.77it/s]


EMB_SCALE=1.000 | cos(HF*scale, TL): min=1.0000 | norm ratio: [1.000, 1.000]


### 3.3.b Compute the edit: which tokens are edited, and by how much

For each eligible token $t$, the cell computes $\mathbf c_t = \sum_{i\in F_C} Y_{i,t}\mathbf z_i$ and the relative edit magnitude
$\mu_t = \|\mathbf c_t\| / \|\mathbf e_t\|$ (App. F.1). The paper reports that EMBER edits at most ~0.03% (Gemma)
or ~0.14% (Llama) of the vocabulary.

In [ ]:
Y_C = Y[F_C]          # (|F_C|, |V*|)
Z_C = Z[:, F_C]       # (d, |F_C|)

touched = (Y_C.abs() > 0).any(dim=0).numpy()
eligible = touched & (codes == 1)
edit_cols = torch.as_tensor(np.where(eligible)[0], dtype=torch.long)
edit_ids = [vstar_ids[j] for j in edit_cols.tolist()]

C = (Z_C @ Y_C[:, edit_cols]).T                        # (n_edit, d): c_t per edited token
mu = C.norm(dim=1) / E_vstar_tl[edit_cols].norm(dim=1)

edit_table = pd.DataFrame({
    "token": hf_tok.convert_ids_to_tokens(edit_ids),
    "token_id": edit_ids,
    "mu": mu.numpy(),
}).sort_values("mu", ascending=False, ignore_index=True)

print(f"tokens touched by F_C: {touched.sum()}  ->  edited (concept-only): {len(edit_ids)} "
      f"= {len(edit_ids) / VOCAB_SIZE:.4%} of the vocabulary")
skipped = [vstar_ids[j] for j in np.where(touched & (codes != 1))[0]]
print(f"skipped (labelled neutral/both): {hf_tok.convert_ids_to_tokens(skipped)}")
edit_table

tokens touched by F_C: 0  ->  edited (concept-only): 0 = 0.0000% of the vocabulary
skipped (labelled neutral/both): []


,token,token_id,mu


### 3.3.c Build the erased model: $\mathbf e_t \leftarrow \mathbf e_t - \delta\,\mathbf c_t$

Builds a copy of the model that owns only its **input embedding matrix** (≈1 GB) and shares every other tensor
(layers, norms, `lm_head`) with `original_model`, then applies the edit to its embedding rows only; `original_model`
is never touched. The edit is computed in the factorization's scaled space: read `W_E × scale`, subtract,
then divide by `scale`.

`set_erasure_strength(delta)` always rewrites the edited rows from the original values, so you can change
$\delta$ without reloading (the sweep in 3.3.f does this). `set_erasure_strength(0.0)` makes the two models identical.

Memory: two full bf16 copies of Llama-3.1-8B would be 32 GB; sharing makes the second one ≈1 GB.


In [ ]:
import copy


def share_all_but_embeddings(base):
    """A copy of `base` that owns only its input-embedding matrix; every other tensor is shared with `base`."""
    emb = base.get_input_embeddings().weight
    shared = {id(t): t for t in [*base.parameters(), *base.buffers()] if t is not emb}
    return copy.deepcopy(base, memo=shared)


erased_model = share_all_but_embeddings(original_model)

W_erased = erased_model.get_input_embeddings().weight
edit_idx = torch.tensor(edit_ids, dtype=torch.long)
with torch.no_grad():
    original_rows = W_orig.index_select(0, edit_idx.to(W_orig.device)).cpu().clone()


def apply_ember_edit(W: torch.Tensor, delta: float) -> None:
    """Set the edited rows of embedding matrix W to e_t - delta * c_t (always from the original rows)."""
    with torch.no_grad():
        e = original_rows.to(W.device).float() * EMB_SCALE
        e_new = e - delta * C.to(device=W.device, dtype=torch.float32)
        W.index_copy_(0, edit_idx.to(W.device), (e_new / EMB_SCALE).to(W.dtype))


def set_erasure_strength(delta: float) -> None:
    apply_ember_edit(W_erased, delta)


set_erasure_strength(DELTA)

# Sanity check: only the edited embedding rows differ, and the unembedding is identical.
with torch.no_grad():
    changed = (W_orig.to(W_erased.device) != W_erased).any(dim=1).nonzero().flatten().tolist()
lm_head_same = torch.equal(original_model.get_output_embeddings().weight,
                           erased_model.get_output_embeddings().weight)
print(f"delta={DELTA}: {len(changed)} embedding rows differ ({len(edit_ids)} eligible); "
      f"lm_head identical: {lm_head_same}")
assert set(changed) <= set(edit_ids) and lm_head_same


delta=10.0: 0 embedding rows differ (0 eligible); lm_head identical: True


### 3.3.d Query API: original vs. erased

| Function | What it does |
|---|---|
| `ask(question, model="original" or "erased")` | one greedy answer from one model (chat template applied) |
| `compare(questions)` | prints each question with both models' answers one under the other |
| `mc_accuracy(model)` | the paper's MC protocol on the test split: concept and similar-domain accuracy |
| `compare_mc()` | `mc_accuracy` for both models in one table |


In [ ]:
import textwrap

MODELS = {
    "original": WrappedHFModel(original_model, hf_tok),
    "erased": WrappedHFModel(erased_model, hf_tok),
}

mc_sets = {
    "concept": prepare_mc_items(load_mc_qa_items("qa_test", CONCEPT), seed=SEED),
    "similar-domain": prepare_mc_items(load_mc_qa_items("simdom_test", CONCEPT), seed=SEED),
}


def ask(question: str, model: str = "erased", max_new_tokens: int = 60) -> str:
    m = MODELS[model]
    return m.generate(m.wrap_prompt(question), max_new_tokens=max_new_tokens, do_sample=False)


def compare(questions, max_new_tokens: int = 60) -> None:
    for q in [questions] if isinstance(questions, str) else questions:
        print(f"Q: {q}")
        for name in MODELS:
            answer = textwrap.indent(ask(q, name, max_new_tokens), " " * 12).lstrip()
            print(f"  {name:<8}: {answer}")
        print()


def mc_accuracy(model: str) -> dict:
    out = {}
    for name, items in mc_sets.items():
        correct, invalid, _ = evaluate_mc_generation(MODELS[model], items)
        out[name] = correct / len(items)
        out[f"{name}_invalid"] = invalid
    return out


def compare_mc() -> pd.DataFrame:
    return pd.DataFrame({name: mc_accuracy(name) for name in MODELS})


In [ ]:
ask("Who was Julius Caesar?", model="original")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'Julius Caesar was a Roman general, statesman, and politician who played a critical role in transforming the Roman Republic into the Roman Empire. He is considered one of the most influential leaders in Roman history.\n\nEarly Life and Career:\nJulius Caesar was born on July 12 or 13,'

In [ ]:
ask("Who was Julius Caesar?", model="erased")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'Julius Caesar was a Roman general, statesman, and politician who played a critical role in transforming the Roman Republic into the Roman Empire. He is considered one of the most influential leaders in Roman history.\n\nEarly Life and Career:\nJulius Caesar was born on July 12 or 13,'

In [ ]:
ask("What was Ancient Rome?", model="erased")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


"Ancient Rome was a civilization that existed from 8th century BC to the 5th century AD. It was a powerful and influential state that played a significant role in shaping Western civilization. Here's an overview:\n\n**Founding and Early History**\n\nAncient Rome was founded in 753 BC"

In [ ]:
ask("What is the capital of Italy?", model="erased")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'The capital of Italy is Rome.'

In [ ]:
ask("What was the Roman Empire in antiquity?", model="erased")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'The Roman Empire was a vast and powerful state that existed from 27 BC to 476 AD in antiquity. It was the successor to the Roman Republic, which had been a system of governance by the Senate and the Assemblies. The Roman Empire was a major power in the ancient world, known'

In [ ]:
ask("What is the capital of Italy?", model="original")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'The capital of Italy is Rome.'

In [ ]:
ask("What is modern-day Rome known for?", model="original")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'Modern-day Rome, the capital city of Italy, is known for its rich history, architecture, art, and culture. Here are some of the top things Rome is known for:\n\n1. **Ancient Ruins and Landmarks**: Rome is home to numerous ancient ruins, including the Colosse'

In [ ]:
ask("What is modern-day Rome known for?", model="erased")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'Modern-day Rome, the capital city of Italy, is known for its rich history, architecture, art, and culture. Here are some of the top things Rome is known for:\n\n1. **Ancient Ruins and Landmarks**: Rome is home to numerous ancient ruins, including the Colosse'

### 3.3.g One erased model per $\delta$ in the grid
Builds an erased model for every $\delta$ in `DELTA_GRID` and registers each one in `MODELS` as `delta_<δ>`
(`delta_0.5`, `delta_1`, …, `delta_200`). You can then query any of them with the 3.3.d API:
`ask("Who was Julius Caesar?", model="delta_50")`.

**Memory.** Eight full copies would not fit (about 16 GB each for Llama-3.1-8B).
The edit changes only rows of the input embedding matrix, so each copy owns its **own embedding matrix**
(about 1.05 GB: 128,256 × 4096 in bf16) and shares every other tensor (layers, norms, `lm_head`) with `original_model`.
Nothing ever writes to the shared tensors, so this is safe. The cell checks free RAM before it builds anything.

`compare()` and `compare_mc()` loop over all of `MODELS`, so after this cell they include the 8 extra models
too (slow on CPU). `compare_deltas()` prints `original` plus every `delta_<δ>` answer for a question.

In [ ]:
import psutil

def delta_key(delta: float) -> str:
    return f"delta_{delta:g}"

todo = [d for d in DELTA_GRID if delta_key(d) not in MODELS]
emb_bytes = W_orig.numel() * W_orig.element_size()
full_bytes = sum(p.numel() * p.element_size() for p in original_model.parameters())
need, avail = len(todo) * emb_bytes, psutil.virtual_memory().available
print(f"full copy: {full_bytes / 2**30:.1f} GiB, so {len(DELTA_GRID)} full copies would need "
      f"{len(DELTA_GRID) * full_bytes / 2**30:.1f} GiB")
print(f"shared copy: {emb_bytes / 2**30:.2f} GiB each, so {len(todo)} to build need {need / 2**30:.1f} GiB "
      f"| available now: {avail / 2**30:.1f} GiB")
assert need < 0.8 * avail, "not enough free RAM for one embedding matrix per delta"

for delta in tqdm(todo, desc="building per-delta models"):
    m = share_all_but_embeddings(original_model)
    apply_ember_edit(m.get_input_embeddings().weight, delta)
    MODELS[delta_key(delta)] = WrappedHFModel(m, hf_tok)

# Sanity check: each copy owns only its embedding matrix and shares lm_head with the original.
orig_ptrs = {n: p.data_ptr() for n, p in original_model.named_parameters()}
for delta in DELTA_GRID:
    m = MODELS[delta_key(delta)].model
    own = [n for n, p in m.named_parameters() if p.data_ptr() != orig_ptrs[n]]
    assert own == ["model.embed_tokens.weight"], own
print(f"\nqueryable: {[delta_key(d) for d in DELTA_GRID]}   e.g. ask(q, model={delta_key(DELTA_GRID[-1])!r})")

def compare_deltas(questions, deltas=None, max_new_tokens: int = 60) -> None:
    """Prints the original model's answer and every per-delta model's answer to each question."""
    names = ["original"] + [delta_key(d) for d in (deltas or DELTA_GRID)]
    for q in [questions] if isinstance(questions, str) else questions:
        print(f"Q: {q}")
        for name in names:
            answer = textwrap.indent(ask(q, name, max_new_tokens), " " * 14).lstrip()
            print(f"  {name:<10}: {answer}")
        print()

full copy: 15.0 GiB, so 8 full copies would need 119.7 GiB
shared copy: 0.98 GiB each, so 8 to build need 7.8 GiB | available now: 45.7 GiB


building per-delta models: 100%|██████████| 8/8 [00:00<00:00, 14.76it/s]


queryable: ['delta_0.5', 'delta_1', 'delta_2', 'delta_5', 'delta_10', 'delta_50', 'delta_100', 'delta_200']   e.g. ask(q, model='delta_200')


In [ ]:
DELTA_GRID = [0.5, 1.0, 2.0, 5.0, 10.0]

In [ ]:
compare_deltas("Who was Julius Caesar?")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: Who was Julius Caesar?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  original  : Julius Caesar was a Roman general, statesman, and politician who played a critical role in transforming the Roman Republic into the Roman Empire. He is considered one of the most influential leaders in Roman history.

              Early Life and Career:
              Julius Caesar was born on July 12 or 13,


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

  delta_1   : Julius Caesar was a Roman general, statesman, and politician who played a critical role in transforming the Roman Republic into the Roman Empire. He is considered one of the most influential leaders in Roman history.

              Early Life and Career:
              Julius Caesar was born on July 12 or 13,


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  delta_2   : Julius Caesar was a Roman general, statesman, and politician who played a critical role in transforming the Roman Republic into the Roman Empire. He is considered one of the most influential leaders in Roman history.

              Early Life and Career:
              Julius Caesar was born on July 12 or 13,


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  delta_5   : Julius Caesar was a Roman general, statesman, and politician who played a critical role in transforming the Roman Republic into the Roman Empire. He is considered one of the most influential leaders in Roman history.

              Early Life and Career:
              Julius Caesar was born on July 12 or 13,
  delta_10  : Julius Caesar was a Roman general, statesman, and politician who played a critical role in transforming the Roman Republic into the Roman Empire. He is considered one of the most influential leaders in Roman history.

              Early Life and Career:
              Julius Caesar was born on July 12 or 13,



In [ ]:
compare_deltas("Who was Julius Caesar?")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: Who was Julius Caesar?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  original  : Julius Caesar was a Roman general, statesman, and politician who played a critical role in transforming the Roman Republic into the Roman Empire. He is considered one of the most influential leaders in Roman history.

              Early Life and Career:
              Julius Caesar was born on July 12 or 13,


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


...

  delta_1   : Julius Caesar was a Roman general, statesman, and politician who played a critical role in transforming the Roman Republic into the Roman Empire. He is considered one of the most influential leaders in Roman history.

              Early Life and Career:
              Julius Caesar was born on July 12 or 13,


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  delta_2   : Julius Caesar was a Roman general, statesman, and politician who played a critical role in transforming the Roman Republic into the Roman Empire. He is considered one of the most influential leaders in Roman history.

              Early Life and Career:
              Julius Caesar was born on July 12 or 13,


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  delta_5   : Julius Caesar was a Roman general, statesman, and politician who played a critical role in transforming the Roman Republic into the Roman Empire. He is considered one of the most influential leaders in Roman history.

              Early Life and Career:
              Julius Caesar was born on July 12 or 13,
  delta_10  : Julius Caesar was a Roman general, statesman, and politician who played a critical role in transforming the Roman Republic into the Roman Empire. He is considered one of the most influential leaders in Roman history.

              Early Life and Career:
              Julius Caesar was born on July 12 or 13,



In [ ]:
compare_deltas("What was Ancient Rome?")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Q: What was Ancient Rome?


KeyboardInterrupt: 

In [ ]:
compare_deltas("What was the Roman Empire in antiquity?")

In [ ]:
compare_deltas("What is the capital of Italy?")

In [ ]:
compare_deltas("What is modern-day Rome known for?")

In [ ]:
DELTA_GRID = [1.0, 1.25, 1.5, 1.75, 2.0]

Question = [
    "Who was Julius Caesar?",
    "What was Ancient Rome?",
    "What was the Roman Empire in antiquity?",
    "What is the capital of Italy?",
    "What is modern-day Rome known for?",
]

for question in Question:
    compare_deltas(question)

---
# 3.4 Tune $\delta$ with the paper's grid search (Stage 1, App. C.2)

The paper tunes EMBER's $\delta$ per concept in isolation ("Stage 1: Embedding grid"). This section
reproduces that protocol as implemented in `ember/erasure/methods/ember.py`, `ember/evals/orchestrator.py`
and `io.pick_best_embed_delta`:

| Item | Value |
|---|---|
| Grid | $\delta \in$ {0.5, 1, 2, 5, 10, 50, 100, 200} ($\delta=0$ is not evaluated) |
| Split | **validation** (`train` in the repo): 50 concept MC, 50 similar-domain MC, 50 MMLU, 150 AlpacaEval prompts |
| Normalization | MC and MMLU: $\text{clip}\big(\tfrac{\text{acc}-0.25}{\text{base}-0.25},0,1\big)$. Alpaca: $\min(1, \text{score}/\text{base})$. The base is the original model on the same split. |
| Axes | efficacy $= 1-\widehat{\text{qa}}$; specificity $= \text{HM}(\widehat{\text{mmlu}}, \widehat{\text{simdom}})$; coherence $= \text{HM}(\widehat{\text{instr}}, \widehat{\text{flu}})$ |
| Score | $H = \text{HM}(\text{efficacy}, \text{specificity}, \text{coherence})$ |
| Gates | $\widehat{\text{mmlu}} < 0.9 \Rightarrow H=0$ (the rest is skipped). $\widehat{\text{instr}}$ or $\widehat{\text{flu}} < 0.9 \Rightarrow H=0$. |
| Pick | largest $H$; ties go to the smaller $\delta$; if every $H=0$ then $\delta=0$ (no edit) |

The grid runs on a **third model copy, `tuned`**, so `original` and `erased` are untouched. At the end `tuned`
holds the best $\delta$ and is added to the query API.

**Cost.** AlpacaEval dominates: for each $\delta$ that passes the MMLU gate, 150 generations of up to
200 tokens plus 300 Gemini judge calls. This is slow on CPU. `TUNE_WITH_ALPACA = False` drops the coherence
axis, so $H = \text{HM}(\text{efficacy}, \text{specificity})$. That is faster but **not** the paper's protocol.
Memory: `tuned` shares everything but its embedding matrix with `original_model`, so it adds ≈1 GB.

Tuning uses the validation split; `compare_mc()` reports on the held-out test split, so there is no leakage.


In [ ]:
import threading

from ember.evals.alpaca import evaluate_alpaca
from ember.evals.gemini import GeminiEvaluator
from ember.evals.harmonic import harmonic_mean
from ember.evals.mmlu import evaluate_mmlu_generation, load_mmlu_items
from ember.local_datasets import load_alpaca_indices, load_mmlu_indices

TUNE_WITH_ALPACA = True   # paper protocol; False drops the coherence axis (faster, not the paper's H)
MIN_MMLU = 0.90           # repo EMBER gates (EMBERMethod.grid_eval_kwargs)
MIN_ALPACA = 0.90
ALPACA_BATCH = 8

val_mc = {
    "qa": prepare_mc_items(load_mc_qa_items("qa_train", CONCEPT), seed=SEED),
    "simdom": prepare_mc_items(load_mc_qa_items("simdom_train", CONCEPT), seed=SEED),
}
val_mmlu = load_mmlu_items(load_mmlu_indices("train"))
N_ALPACA = len(load_alpaca_indices("train"))
judge = GeminiEvaluator(model_name=GEMINI_MODEL) if TUNE_WITH_ALPACA else None


class ProgressModel:
    """Forwards to a WrappedHFModel and advances `bar` once per generated prompt."""

    def __init__(self, inner, bar):
        self.inner, self.bar = inner, bar

    def __getattr__(self, name):
        return getattr(self.inner, name)

    def generate(self, *args, **kwargs):
        out = self.inner.generate(*args, **kwargs)
        self.bar.update(1)
        return out

    def generate_multiple(self, prompts, *args, batch_size=50, **kwargs):
        out = []
        for i in range(0, len(prompts), batch_size):
            chunk = prompts[i:i + batch_size]
            out += self.inner.generate_multiple(chunk, *args, batch_size=batch_size, **kwargs)
            self.bar.update(len(chunk))
        return out


class ProgressJudge:
    """Forwards to a GeminiEvaluator and advances `bar` once per `score_*` call."""

    def __init__(self, inner, bar):
        self.inner, self.bar, self.lock = inner, bar, threading.Lock()

    def __getattr__(self, name):
        attr = getattr(self.inner, name)
        if not name.startswith("score_"):
            return attr

        def scored(*args, **kwargs):
            out = attr(*args, **kwargs)
            with self.lock:  # evaluate_alpaca scores from 30 threads
                self.bar.update(1)
            return out
        return scored


def eval_total(with_alpaca: bool) -> int:
    """Progress-bar steps for one full validation pass (Alpaca = 1 generation + 2 judge calls per item)."""
    n = len(val_mmlu) + len(val_mc["qa"]) + len(val_mc["simdom"])
    return n + 3 * N_ALPACA if with_alpaca else n


def mc_norm(acc: float, base: float) -> float:
    return min(1.0, max((acc - 0.25) / (base - 0.25), 0.0)) if base != 0.25 else 0.0


def mmlu_acc(wrapped) -> float:
    correct, _, _ = evaluate_mmlu_generation(wrapped, val_mmlu)
    return correct / len(val_mmlu)


def mc_acc(wrapped, name: str) -> float:
    correct, _, _ = evaluate_mc_generation(wrapped, val_mc[name])
    return correct / len(val_mc[name])


def alpaca_scores(wrapped, judge_=None) -> tuple:
    instr, flu, _ = evaluate_alpaca(wrapped, judge_ or judge, "train", batch_size=ALPACA_BATCH)
    return float(np.mean(instr)), float(np.mean(flu))


# Baseline = the original model on the validation split (computed once, reused on re-runs).
if "val_base" not in globals() or (TUNE_WITH_ALPACA and "alpaca_instr" not in val_base):
    with tqdm(total=eval_total(TUNE_WITH_ALPACA), desc="baseline (original)", unit="step") as bar:
        m = ProgressModel(MODELS["original"], bar)
        bar.set_postfix_str("MMLU")
        val_base = {"mmlu": mmlu_acc(m)}
        bar.set_postfix_str("concept MC")
        val_base["qa"] = mc_acc(m, "qa")
        bar.set_postfix_str("similar-domain MC")
        val_base["simdom"] = mc_acc(m, "simdom")
        if TUNE_WITH_ALPACA:
            bar.set_postfix_str("AlpacaEval")
            val_base["alpaca_instr"], val_base["alpaca_flu"] = alpaca_scores(m, ProgressJudge(judge, bar))
val_base


In [ ]:
def h_score(wrapped) -> dict:
    """Stage-1 H score of one edited model, with the repo's early-stop gates."""
    row = {"mmlu_acc": mmlu_acc(wrapped)}
    row["mmlu_frac"] = mc_norm(row["mmlu_acc"], val_base["mmlu"])
    if row["mmlu_frac"] < MIN_MMLU:
        return {**row, "H": 0.0, "gate": "mmlu"}

    for name in ("qa", "simdom"):
        row[f"{name}_acc"] = mc_acc(wrapped, name)
        row[f"{name}_frac"] = mc_norm(row[f"{name}_acc"], val_base[name])
    row["efficacy"] = 1.0 - row["qa_frac"]
    row["specificity"] = harmonic_mean([row["mmlu_frac"], row["simdom_frac"]])
    axes = [row["efficacy"], row["specificity"]]

    if TUNE_WITH_ALPACA:
        instr, flu = alpaca_scores(wrapped)
        row["instr_frac"] = min(1.0, instr / val_base["alpaca_instr"])
        row["flu_frac"] = min(1.0, flu / val_base["alpaca_flu"])
        row["coherence"] = harmonic_mean([row["instr_frac"], row["flu_frac"]])
        axes.append(row["coherence"])
        if row["instr_frac"] < MIN_ALPACA or row["flu_frac"] < MIN_ALPACA:
            return {**row, "H": 0.0, "gate": "alpaca"}

    return {**row, "H": harmonic_mean(axes), "gate": ""}


# Fallback for kernels where 3.3.c ran before apply_ember_edit existed (same code as 3.3.c).
if "apply_ember_edit" not in globals():
    edit_idx = torch.tensor(edit_ids, dtype=torch.long)
    with torch.no_grad():
        original_rows = W_orig.index_select(0, edit_idx.to(W_orig.device)).cpu().clone()

    def apply_ember_edit(W: torch.Tensor, delta: float) -> None:
        """Set the edited rows of embedding matrix W to e_t - delta * c_t (always from the original rows)."""
        with torch.no_grad():
            e = original_rows.to(W.device).float() * EMB_SCALE
            e_new = e - delta * C.to(device=W.device, dtype=torch.float32)
            W.index_copy_(0, edit_idx.to(W.device), (e_new / EMB_SCALE).to(W.dtype))


if "tuned_model" not in globals():
    tuned_model = share_all_but_embeddings(original_model)
W_tuned = tuned_model.get_input_embeddings().weight
MODELS["tuned"] = WrappedHFModel(tuned_model, hf_tok)

grid_rows = []
for delta in tqdm(DELTA_GRID, desc="Stage 1: embedding grid"):
    apply_ember_edit(W_tuned, delta)
    with Timer() as t:
        row = h_score(MODELS["tuned"])
    grid_rows.append({"delta": delta, **row, "seconds": round(t["elapsed"], 1)})
    print(f"delta={delta:>6}: H={row['H']:.3f}  {('gated by ' + row['gate']) if row['gate'] else ''}")

grid = pd.DataFrame(grid_rows)
passed = grid[grid["H"] > 0]
# Same rule as io.pick_best_embed_delta: max H, ties -> smaller delta, all zero -> no edit.
BEST_DELTA = 0.0 if passed.empty else float(
    passed.sort_values(["H", "delta"], ascending=[False, True]).iloc[0]["delta"])
apply_ember_edit(W_tuned, BEST_DELTA)

print(f"\nBEST_DELTA = {BEST_DELTA}" + ("  (every H was 0, so no edit)" if passed.empty else ""))
grid


### 3.4.b Compare all three models

`MODELS` now holds `original`, `erased` ($\delta$ = `DELTA`) and `tuned` ($\delta$ = `BEST_DELTA`), so the
3.3.d API covers all of them: `ask(q, model="tuned")`, `compare([...])`, `compare_mc()`.
`compare_mc()` reports on the **test** split, which the grid never saw.


In [ ]:
print(f"erased: delta={DELTA}   tuned: delta={BEST_DELTA}\n")
compare(concept_qs + simdom_qs)


In [ ]:
mc_table = compare_mc()
mc_table


In [ ]:
# Check if the one feature direction of the concept belongs to the vector space spanned by the other feature directions
# z_i is in span{z_j : j != i} iff its least-squares projection onto that span leaves a ~0 residual.
# Reference point: with k-1 = 199 directions in d = 4096 dims, a random direction keeps ~sqrt(1 - 199/4096) ≈ 0.975
# of its norm, so a residual near 0.98 means "no more in the span than a random vector".
Z64 = Z.double()
d, k = Z64.shape


def span_residual(z: torch.Tensor, B: torch.Tensor):
    """Relative residual ||z - P_B z|| / ||z|| and the least-squares coefficients of z on the columns of B."""
    coef = torch.linalg.lstsq(B, z.unsqueeze(1), driver="gelsd").solution.squeeze(1)
    return ((z - B @ coef).norm() / z.norm()).item(), coef


rows = []
for i in range(k):
    others = [j for j in range(k) if j != i]
    resid, _ = span_residual(Z64[:, i], Z64[:, others])
    rows.append({"feature": i, "resid_frac": resid, "in_F_C": i in F_C})
span_df = pd.DataFrame(rows).sort_values("resid_frac").reset_index(drop=True)

g = torch.Generator().manual_seed(SEED)
rand_resid = [span_residual(torch.randn(d, generator=g, dtype=torch.float64), Z64[:, 1:])[0] for _ in range(200)]

print(f"rank(Z) = {torch.linalg.matrix_rank(Z64).item()} / {k}   cond(Z) = {torch.linalg.cond(Z64).item():.1f}")
print(f"random-direction residual: mean={np.mean(rand_resid):.4f}  min={np.min(rand_resid):.4f}")
print(f"other features' residual  : median={span_df.loc[~span_df.in_F_C, 'resid_frac'].median():.4f}  "
      f"min={span_df.loc[~span_df.in_F_C, 'resid_frac'].min():.4f}\n")

for feat in F_C:
    others = [j for j in range(k) if j != feat]
    resid, coef = span_residual(Z64[:, feat], Z64[:, others])
    rank_pos = int(span_df.index[span_df.feature == feat][0]) + 1
    verdict = "IN the span" if resid < 1e-6 else ("mostly in the span" if resid < 0.5 else "NOT in the span")
    print(f"feature {feat}: residual = {resid:.4f}  (||proj|| / ||z|| = {np.sqrt(max(0.0, 1 - resid**2)):.4f}), "
          f"rank {rank_pos}/{k} from most-in-span  ->  {verdict}")

    # Which other features carry the projection (|coef_j| * ||z_j|| = size of each term in the combination).
    contrib = (coef.abs() * Z64[:, others].norm(dim=0)) / Z64[:, feat].norm()
    top = torch.argsort(contrib, descending=True)[:5]
    print("  largest terms: " + ", ".join(f"z_{others[t]} ({contrib[t]:.3f})" for t in top.tolist()))

span_df.head(10)


In [ ]:
print(Z.size())
print(Y.size())
print(RANK)

In [ ]:
# Please fill here

In [ ]:
# For each of token the concept direction appears on - how many feature are activated in the factorization?
# Support of feature i = tokens t with Y_{i,t} != 0. For each such token, count the nonzero entries of its column Y_{:,t}
# (the concept feature itself included) and compare with every token in V*.
# share = |Y_{i,t}| ||z_i|| / sum_j |Y_{j,t}| ||z_j||: how much of the token's feature reconstruction the concept feature carries.
#
# proj = <e_t, z_i / ||z_i||>, the scalar projection on the concept direction (z oriented so its tokens have Y > 0),
# in three normalizations:
#   / avg feature : proj / mean_j |<e_t, z_j / ||z_j||>| over all k feature directions (z_i included)
#   / random dir  : proj / E_u |<e_t, u>| for u uniform on the unit sphere. Exact, no sampling:
#                   E|<e, u>| = ||e|| * Gamma(d/2) / (sqrt(pi) * Gamma((d+1)/2))  (~ ||e|| sqrt(2 / (pi d)))
#   z vs V*       : (proj - mean_s proj_s) / std_s proj_s over all tokens s in V*, i.e. the same direction across tokens
from collections import Counter

active = Y.abs() > 0                     # (k, |V*|)
n_active = active.sum(0).numpy()         # active features per token
z_norms = Z.norm(dim=0)                  # (k,)
rho = stats.set_index("feature")["ratio_abs"]

E_ref = (E_vstar_tl if "E_vstar_tl" in globals() else E_vstar).float().cpu()  # E_vstar is deleted in 3.3.a
proj_all = E_ref @ (Z / z_norms)         # (|V*|, k): projection of every token on every unit feature direction
avg_feat_proj = proj_all.abs().mean(1)   # (|V*|,)
d = Z.shape[0]
rand_proj = E_ref.norm(dim=1) * math.exp(math.lgamma(d / 2) - math.lgamma((d + 1) / 2)) / math.sqrt(math.pi)

for feat in F_C:
    support = torch.nonzero(active[feat]).flatten()
    support = support[torch.argsort(Y[feat, support].abs(), descending=True)].tolist()
    counts = n_active[support]

    sign = 1.0 if Y[feat, support].sum() >= 0 else -1.0
    proj = sign * proj_all[:, feat]
    proj_z = (proj - proj.mean()) / proj.std()

    print(f"=== feature {feat}: on {len(support)} tokens ===")
    print(f"proj on z_{feat} over all of V*: mean={proj.mean():.3f}  std={proj.std():.3f}")
    print(f"features per token: mean={counts.mean():.2f} (all of V*: {n_active.mean():.2f})  "
          f"min={counts.min()}  max={counts.max()}  only this feature: {int((counts == 1).sum())}")
    dist = pd.DataFrame({
        f"feature {feat} tokens": pd.Series(counts).value_counts(normalize=True),
        "all V*": pd.Series(n_active).value_counts(normalize=True),
    }).fillna(0.0).sort_index()
    dist.index.name = "# active features"
    display(dist.style.format("{:.1%}"))

    rows, co = [], Counter()
    for j in support:
        terms = Y[:, j].abs() * z_norms
        others = [i for i in torch.nonzero(active[:, j]).flatten().tolist() if i != feat]
        others.sort(key=lambda i: -terms[i].item())
        co.update(others)
        rows.append({
            "token": tokenizer.convert_ids_to_tokens(vstar_ids[j]),
            "label": LABEL_TAG[codes[j]],
            f"Y[{feat}]": Y[feat, j].item(),
            "n_active": int(n_active[j]),
            "share": (terms[feat] / terms.sum()).item(),
            "proj": proj[j].item(),
            "proj / avg feature": (proj[j] / avg_feat_proj[j]).item(),
            "proj / random dir": (proj[j] / rand_proj[j]).item(),
            "proj z vs V*": proj_z[j].item(),
            "other features (Y)": ", ".join(f"{i} ({Y[i, j].item():+.2f})" for i in others),
        })
    per_token = pd.DataFrame(rows)
    per_token.index += 1
    with pd.option_context("display.max_rows", None):
        display(per_token)

    # Which other features co-occur most often on this feature's tokens.
    co_df = pd.DataFrame([
        {"feature": i, "shared tokens": n, "frac of support": n / len(support),
         "rho": rho.get(i, float("nan")), "in_F_C": i in F_C}
        for i, n in co.most_common(10)
    ])
    print(f"most frequent co-active features on feature {feat}'s tokens:")
    display(co_df)

In [ ]:
# For each of token the concept direction appears on - how many feature are activated in the factorization?
# Support of feature i = tokens t with Y_{i,t} != 0. For each such token, count the nonzero entries of its column Y_{:,t}
# (the concept feature itself included) and compare with every token in V*.
# share = |Y_{i,t}| ||z_i|| / sum_j |Y_{j,t}| ||z_j||: how much of the token's feature reconstruction the concept feature carries.
#
# proj = <e_t, z_i / ||z_i||>, the scalar projection on the concept direction (z oriented so its tokens have Y > 0),
# in three normalizations:
#   / avg feature : proj / mean_j |<e_t, z_j / ||z_j||>| over all k feature directions (z_i included)
#   / random dir  : proj / E_u |<e_t, u>| for u uniform on the unit sphere. Exact, no sampling:
#                   E|<e, u>| = ||e|| * Gamma(d/2) / (sqrt(pi) * Gamma((d+1)/2))  (~ ||e|| sqrt(2 / (pi d)))
#   z vs V*       : (proj - mean_s proj_s) / std_s proj_s over all tokens s in V*, i.e. the same direction across tokens
from collections import Counter

active = Y.abs() > 0                     # (k, |V*|)
n_active = active.sum(0).numpy()         # active features per token
z_norms = Z.norm(dim=0)                  # (k,)
rho = stats.set_index("feature")["ratio_abs"]

E_ref = (E_vstar_tl if "E_vstar_tl" in globals() else E_vstar).float().cpu()  # E_vstar is deleted in 3.3.a
proj_all = E_ref @ (Z / z_norms)         # (|V*|, k): projection of every token on every unit feature direction
avg_feat_proj = proj_all.abs().mean(1)   # (|V*|,)
d = Z.shape[0]
rand_proj = E_ref.norm(dim=1) * math.exp(math.lgamma(d / 2) - math.lgamma((d + 1) / 2)) / math.sqrt(math.pi)

for feat in F_C:
    support = torch.nonzero(active[feat]).flatten()
    support = support[torch.argsort(Y[feat, support].abs(), descending=True)].tolist()
    counts = n_active[support]

    sign = 1.0 if Y[feat, support].sum() >= 0 else -1.0
    proj = sign * proj_all[:, feat]
    proj_z = (proj - proj.mean()) / proj.std()

    print(f"=== feature {feat}: on {len(support)} tokens ===")
    print(f"proj on z_{feat} over all of V*: mean={proj.mean():.3f}  std={proj.std():.3f}")
    print(f"features per token: mean={counts.mean():.2f} (all of V*: {n_active.mean():.2f})  "
          f"min={counts.min()}  max={counts.max()}  only this feature: {int((counts == 1).sum())}")
    dist = pd.DataFrame({
        f"feature {feat} tokens": pd.Series(counts).value_counts(normalize=True),
        "all V*": pd.Series(n_active).value_counts(normalize=True),
    }).fillna(0.0).sort_index()
    dist.index.name = "# active features"
    display(dist.style.format("{:.1%}"))

    rows, co = [], Counter()
    for j in support:
        terms = Y[:, j].abs() * z_norms
        others = [i for i in torch.nonzero(active[:, j]).flatten().tolist() if i != feat]
        others.sort(key=lambda i: -terms[i].item())
        co.update(others)
        rows.append({
            "token": tokenizer.convert_ids_to_tokens(vstar_ids[j]),
            "label": LABEL_TAG[codes[j]],
            f"Y[{feat}]": Y[feat, j].item(),
            "n_active": int(n_active[j]),
            "share": (terms[feat] / terms.sum()).item(),
            "proj": proj[j].item(),
            "proj / avg feature": (proj[j] / avg_feat_proj[j]).item(),
            "proj / random dir": (proj[j] / rand_proj[j]).item(),
            "proj z vs V*": proj_z[j].item(),
            "other features (Y)": ", ".join(f"{i} ({Y[i, j].item():+.2f})" for i in others),
        })
    per_token = pd.DataFrame(rows)
    per_token.index += 1
    with pd.option_context("display.max_rows", None):
        display(per_token)

    # Which other features co-occur most often on this feature's tokens.
    co_df = pd.DataFrame([
        {"feature": i, "shared tokens": n, "frac of support": n / len(support),
         "rho": rho.get(i, float("nan")), "in_F_C": i in F_C}
        for i, n in co.most_common(10)
    ])
    print(f"most frequent co-active features on feature {feat}'s tokens:")
    display(co_df)

In [ ]:
# All tokens in V*, sorted by proj / avg feature (descending). Uses the variables computed in the cell above.
SHOW_ROWS = 100  # None shows all |V*| rows

terms_all = Y.abs() * z_norms[:, None]            # (k, |V*|): size of each term |Y_{j,t}| ||z_j||
token_strs = tokenizer.convert_ids_to_tokens(vstar_ids)

for feat in F_C:
    sign = 1.0 if Y[feat][active[feat]].sum() >= 0 else -1.0
    proj = sign * proj_all[:, feat]
    proj_z = (proj - proj.mean()) / proj.std()

    all_tokens = pd.DataFrame({
        "token": token_strs,
        "label": [LABEL_TAG[c] for c in codes],
        "in_support": active[feat].numpy(),
        # f"Y[{feat}]": Y[feat].numpy(),
        # "n_active": n_active,
        "share": (terms_all[feat] / terms_all.sum(0)).numpy(),
        # "proj": proj.numpy(),
        "proj / avg feature": (proj / avg_feat_proj).numpy(),
        # "proj / random dir": (proj / rand_proj).numpy(),
        "proj z vs V*": proj_z.numpy(),
    }).sort_values("proj / avg feature", ascending=False, ignore_index=True)
    all_tokens.index += 1

    n_sup = int(active[feat].sum())
    print(f"=== feature {feat}: all {len(all_tokens)} tokens of V*, sorted by proj / avg feature ===")
    # print(f"support tokens among the top {n_sup}: {int(all_tokens['in_support'].head(n_sup).sum())} / {n_sup}")
    with pd.option_context("display.max_rows", None):
        display(all_tokens if SHOW_ROWS is None else all_tokens.head(SHOW_ROWS))

In [ ]:
# ---- Erase with the original vs the extended factorization; probe with questions on the AR tokens only Y'_ext has ----
# Models: <fac>_d<delta> for fac in {orig (Y, V*), ext (Y_ext, V*_ext)} and delta in DELTAS. Each is a copy that owns
# only its embedding matrix (share_all_but_embeddings, 3.3.g); delta = 0 is the unedited model, so it aliases "original".
# Same edit rule as 3.3.b/c: tokens touched by F_C and labelled concept-only, e_t <- e_t - delta * c_t in the TL-scaled space.
DELTAS = [0.0, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 7.5, 10.0]
N_QUESTIONS = 4
QUESTIONS = None      # None: Gemini phrases them from the new AR tokens; or set your own list of strings
N_TOKENS_PROMPT = 40  # new AR tokens passed to Gemini
RUN_MC = False        # True: also the MC accuracy table (concept / similar-domain test split) for these models; slow

# 1) AR tokens in the extended support (any feature of F_C) but not in the original support
ar_scale = Z.norm(dim=0)[F_C][:, None]                               # unit-norm convention, comparable across F_C
orig_ar_ids = {vstar_ids[j] for j in torch.nonzero((Y[F_C] != 0).any(0)).flatten().tolist()}
ext_ar_cols = torch.nonzero((Y_ext[F_C] != 0).any(0)).flatten().tolist()
ext_ar_mag = (Y_ext[F_C] * ar_scale).abs().max(0).values
new_ar = sorted((j for j in ext_ar_cols if vstar_ext[j] not in orig_ar_ids), key=lambda j: -ext_ar_mag[j].item())
ext_label = lambda j: LABEL_TAG[int(codes_ext[j])] + ("*" if vstar_ext[j] in new_tokens else "")
print(f"AR support: original {len(orig_ar_ids)} tokens, extended {len(ext_ar_cols)}, "
      f"only in extended {len(new_ar)}:")
print(", ".join(f"{ext_strs[j]!r}[{ext_label(j)}]" for j in new_ar))

# 2) questions around them
if QUESTIONS is None:
    words = [ext_strs[j].replace("▁", "") for j in new_ar if codes_ext[j] == 1][:N_TOKENS_PROMPT] \
            or [ext_strs[j].replace("▁", "") for j in new_ar][:N_TOKENS_PROMPT]
    raw = client.generate(
        f"These word pieces come from the English Wikipedia article on {CONCEPT}: {', '.join(words)}.\n"
        f"Write {N_QUESTIONS} short, factual questions about {CONCEPT} that a knowledgeable person could answer, "
        f"each built around one or more of these terms. Output one question per line, no numbering, no other text.")
    QUESTIONS = [q.strip().lstrip("-*•0123456789.) ").strip() for q in raw.splitlines() if q.strip().endswith("?")]
    QUESTIONS = QUESTIONS[:N_QUESTIONS]
assert QUESTIONS, "no questions; set QUESTIONS by hand"
print("\nquestions:\n" + "\n".join(f"  {i}. {q}" for i, q in enumerate(QUESTIONS, 1)))


# 3) the two edits
def concept_edit(Y_C: torch.Tensor, codes_vec, ids_vec):
    """3.3.b rule: tokens touched by F_C and labelled concept-only; c_t = sum_i Y_it z_i (TL-scaled space)."""
    cols = np.where((Y_C != 0).any(0).numpy() & (np.asarray(codes_vec) == 1))[0]
    return [ids_vec[j] for j in cols], (Z[:, F_C] @ Y_C[:, cols]).T


edits = {"orig": concept_edit(Y[F_C], codes, vstar_ids), "ext": concept_edit(Y_ext[F_C], codes_ext, vstar_ext)}
o_ids, x_ids = set(edits["orig"][0]), set(edits["ext"][0])
print(f"\nedited tokens: orig {len(o_ids)}, ext {len(x_ids)}, both {len(o_ids & x_ids)}")
print(f"  only ext: {hf_tok.convert_ids_to_tokens(sorted(x_ids - o_ids))}")
print(f"  only orig: {hf_tok.convert_ids_to_tokens(sorted(o_ids - x_ids))}")


def apply_edit(W: torch.Tensor, ids, C_rows: torch.Tensor, delta: float) -> None:
    """W <- original embeddings, then rows `ids` <- e_t - delta * c_t (re-run safe: always from W_orig)."""
    idx = torch.tensor(ids, dtype=torch.long, device=W.device)
    with torch.no_grad():
        W.copy_(W_orig.to(W.device))
        e = W_orig.index_select(0, idx.to(W_orig.device)).to(W.device).float() * EMB_SCALE
        W.index_copy_(0, idx, ((e - delta * C_rows.to(W.device, torch.float32)) / EMB_SCALE).to(W.dtype))


# 4) one model per (factorization, delta)
keys = {(fac, d): f"{fac}_d{d:g}" for fac in edits for d in DELTAS}
to_build = [k for (fac, d), k in keys.items() if d != 0 and k not in MODELS]
need, avail = len(to_build) * W_orig.numel() * W_orig.element_size(), psutil.virtual_memory().available
print(f"\nbuilding {len(to_build)} copies: {need / 2**30:.1f} GiB needed, {avail / 2**30:.1f} GiB available")
assert need < 0.8 * avail, "not enough free RAM for one embedding matrix per model"
for (fac, d), key in tqdm(keys.items(), desc="erased models"):
    if d == 0:
        MODELS[key] = MODELS["original"]
        continue
    if key not in MODELS:
        MODELS[key] = WrappedHFModel(share_all_but_embeddings(original_model), hf_tok)
    apply_edit(MODELS[key].model.get_input_embeddings().weight, *edits[fac], d)

for (fac, d), key in keys.items():   # sanity: only that factorization's edited rows differ from the original
    if d == 0:
        continue
    W = MODELS[key].model.get_input_embeddings().weight
    with torch.no_grad():
        changed = set((W != W_orig.to(W.device)).any(1).nonzero().flatten().tolist())
    assert changed <= set(edits[fac][0]), key

# 5) compare: every question, every model (delta = 0 answered once, as "original")
names = ["original"] + [k for (fac, d), k in keys.items() if d != 0]
for q in QUESTIONS:
    print(f"\nQ: {q}")
    for name in names:
        label = "original (= orig_d0 = ext_d0)" if name == "original" else name
        print(f"  {label:<30}: " + textwrap.indent(ask(q, name), " " * 34).lstrip())

if RUN_MC:
    display(pd.DataFrame({name: mc_accuracy(name) for name in names}))


In [ ]:
ids_x, C_x = edits["ext"]
cols_x = [vstar_ext.index(t) for t in ids_x]
C_unit = (Zn[:, F_C] @ Yn_ext[F_C][:, cols_x]).T          # same edit from the unit-norm pair
print(f"max |C_old - C_unit| = {(C_x - C_unit).abs().max():.2e}   (max |C| = {C_x.abs().max():.2e})")
assert torch.allclose(C_x, C_unit, rtol=1e-4, atol=1e-4 * C_x.abs().max().item())


In [ ]:
for q in QUESTIONS:
    q_ids = set(hf_tok(q, add_special_tokens=False)["input_ids"])
    print(f"Q: {q}")
    for fac, (ids, _) in edits.items():
        hit = sorted(q_ids & set(ids))
        print(f"  {fac:<4} edits {len(hit)} of its tokens: {hf_tok.convert_ids_to_tokens(hit)}")


In [ ]:
# ---- Which directions z_i carry given tokens? Sparse support (Yn) and dense refit (Y') in V* and V*_ext ----
WORDS = [" Trajan", "Trajan"]    # tokenized as-is; add more words or pieces here
TOP_DENSE = 5

tok_ids = list(dict.fromkeys(t for w in WORDS for t in hf_tok(w, add_special_tokens=False)["input_ids"]))
print("pieces:", {w: hf_tok.convert_ids_to_tokens(hf_tok(w, add_special_tokens=False)["input_ids"]) for w in WORDS})


def directions(t, ids, codes_vec, Y_sparse, Y_dns, star=False):
    if t not in ids:
        return None
    j = ids.index(t)
    label = LABEL_TAG[int(codes_vec[j])] + ("*" if star and t in new_tokens else "")
    sup = torch.nonzero(Y_sparse[:, j]).flatten().tolist()
    sup.sort(key=lambda f: -Y_sparse[f, j].abs().item())
    dense = torch.argsort(Y_dns[:, j].abs(), descending=True)[:TOP_DENSE].tolist()
    return label, [(f, Y_sparse[f, j].item()) for f in sup], [(f, Y_dns[f, j].item()) for f in dense]


rows = []
for t in tok_ids:
    s = hf_tok.convert_ids_to_tokens([t])[0]
    print(f"\n=== {s!r} (id {t}) ===")
    for name, res in [("V*  (3.1 fit)", directions(t, vstar_ids, codes, Yn, Y_dense)),
                      ("V*_ext (fixed Z)", directions(t, vstar_ext, codes_ext, Yn_ext, Yd_ext, star=True))]:
        if res is None:
            print(f"  {name}: token not in this set")
            continue
        label, sup, dense = res
        print(f"  {name}: label [{label}], {len(sup)} direction(s) in the sparse support")
        for f, v in sup:
            mark = "   <-- AR" if f in F_C else ""
            print(f"     sparse  z_{f:<3d} Yn = {v:+8.3f}   {all_descriptions.get(f, '')[:90]}{mark}")
        for r, (f, v) in enumerate(dense, 1):
            mark = "   <-- AR" if f in F_C else ""
            print(f"     dense#{r} z_{f:<3d} Y' = {v:+8.3f}   {all_descriptions.get(f, '')[:90]}{mark}")
        for f in F_C:
            if f not in [g for g, _ in dense]:
                print(f"     dense   z_{f:<3d} Y' = {(Y_dense if 'ext' not in name else Yd_ext)[f, (vstar_ids if 'ext' not in name else vstar_ext).index(t)]:+8.3f}   (AR, outside top {TOP_DENSE})")
